# Human Capture — Video → Common Behavior Data

**Demo A of the [Common Behavior Data](https://github.com/Koichi3333/common-behavior-data) project.**

One ordinary video of a person. One reusable behavior dataset. Several very
different renderers reading from it.

> **One source video becomes one reusable behavior dataset —
> for avatars, simulation, analytics, and Physical AI.**

The interesting part is not the motion capture. It is the arrow in the middle:

```text
Video ──▶ MediaPipe ──▶ Common Behavior Data ──┬──▶ MediaPipe overlay
                        (the master data)      ├──▶ MuJoCo   (humanoid.xml + motion.npz)
                                               ├──▶ Unity/VRM (motion.vrma)
                                               └──▶ Behavior dataset (frames.jsonl, CSVs)
```

MediaPipe is never wired straight to MuJoCo or to Unity. Everything downstream
reads the same engine-independent representation, so adding a fourth consumer
means writing one adapter, not one more pipeline.

## What you get

| # | Output | File |
|---|---|---|
| 1 | MediaPipe overlay video | `output/01_mediapipe_overlay/mediapipe_overlay.mp4` |
| 2 | MuJoCo output | `output/02_mujoco/` — `humanoid.xml` + `motion.npz` + replay script + mp4 |
| 3 | Unity / VRM output | `output/03_unity_vrm/motion.vrma` (record the video locally in Unity) |
| 4 | **Behavior dataset** | `output/04_behavior_dataset/` — the master data |
| + | Temporal captions | `04_behavior_dataset/captions/` (optional, Gemini API) |
| + | Motion comparison | `output/05_comparison/motion_comparison.mp4` |
| + | Bundle | `demo2_output_bundle.zip` — feed this straight into Demo B |

## How to run

1. `Runtime → Change runtime type → T4 GPU` (CPU also completes, just slower)
2. *(optional)* Add a Colab secret named `GEMINI_API_KEY` to get temporal
   captions in `[5.5]` — the language half of the dataset
3. Run cells `[1]` → `[10]` in order. `[2]` asks for your video.
4. *(optional)* Play `motion.vrma` in Unity, record it, drop
   `unity_vrm_animation.mp4` back in and re-run `[9]` for the full
   three-screen comparison

**A good source video:** one person, full body in frame, hands visible,
picking up and putting down a bottle / cup / box, 10–30 seconds, ~30fps,
fixed camera. Anything missing is simply omitted and the reason is recorded
in `quality/quality.json` — the run does not stop.

## Honest labelling

- Interaction events are **heuristic candidates**, not ground truth.
- MuJoCo playback is **kinematic replay** (`qpos` assignment + `mj_forward`).
  It is not a claim about physically correct contact or forces.
- Object 3D positions are monocular estimates and always carry a
  `position_source` field.
- Captions are **AI-generated descriptions**, not verified annotations.


In [ ]:
# =====================================================================
# [0] Optional: mount Google Drive
#     Only needed if your source video lives in Drive. Colab upload works
#     just as well -- leave this False and cell [2] will prompt you.
# =====================================================================
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
# =====================================================================
# [1] Environment setup + model download
#     Run once per runtime. A T4 GPU is recommended, but CPU completes too
#     (it is simply slower -- nothing in this notebook requires a GPU).
# =====================================================================
import os
import subprocess
import sys
import time as _time

SETUP_START = _time.time()

print("--- Installing packages (takes 1-3 minutes) ---")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "mediapipe", "mujoco", "imageio-ffmpeg"],
    check=False,
)

# ---------------------------------------------------------------
# Detect the GPU and pick a MuJoCo offscreen rendering backend
#   GPU runtime -> egl    (we generate the EGL ICD config file ourselves)
#   CPU runtime -> osmesa (software rendering)
# ---------------------------------------------------------------
gpu_query = subprocess.run(
    "nvidia-smi --query-gpu=name --format=csv,noheader",
    shell=True, capture_output=True, text=True,
)
HAS_GPU = gpu_query.returncode == 0
GPU_NAME = gpu_query.stdout.strip().splitlines()[0] if HAS_GPU and gpu_query.stdout.strip() else None

if HAS_GPU:
    # Colab ships without an EGL ICD config. Without it MuJoCo aborts the
    # whole process with mju_error and the kernel restarts -- writing this
    # tiny JSON file is the fix.
    NVIDIA_ICD_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_PATH), exist_ok=True)
        with open(NVIDIA_ICD_PATH, "w") as icd_file:
            icd_file.write(
                '{"file_format_version":"1.0.0",'
                '"ICD":{"library_path":"libEGL_nvidia.so.0"}}'
            )
        print("Created EGL ICD config:", NVIDIA_ICD_PATH)
    os.environ["MUJOCO_GL"] = "egl"
    print(f"GPU: {GPU_NAME} / renderer: egl")
else:
    subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null",
                   shell=True, check=False)
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
    print("No GPU / renderer: osmesa (CPU, roughly 5-10x slower)")
    print("Tip: Runtime > Change runtime type > T4 GPU")

import cv2
import mediapipe as mp
import mujoco
import numpy as np

print("mediapipe:", mp.__version__)
print("mujoco   :", mujoco.__version__)

# ---------------------------------------------------------------
# Download the MediaPipe Vision Task models (with URL fallbacks)
#   Five models: Pose / Hand / Face / Gesture / Object
# ---------------------------------------------------------------
import urllib.request
from pathlib import Path

MODEL_DIR = Path("/content/mp_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def download_with_fallback(url_candidates, destination):
    """Try each URL in order and keep the first one that downloads."""
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"  cached: {destination.name}")
        return str(destination)
    last_error = None
    for url in url_candidates:
        try:
            urllib.request.urlretrieve(url, destination)
            if destination.stat().st_size > 0:
                print(f"  OK: {destination.name}")
                return str(destination)
        except Exception as error:  # noqa: BLE001
            last_error = error
    print(f"  FAILED: could not fetch {destination.name} ({last_error})")
    return None


BASE = "https://storage.googleapis.com/mediapipe-models"
print("\n--- Downloading MediaPipe models ---")

POSE_MODEL_PATH = download_with_fallback(
    [f"{BASE}/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
     f"{BASE}/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
     f"{BASE}/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"],
    MODEL_DIR / "pose_landmarker.task")

HAND_MODEL_PATH = download_with_fallback(
    [f"{BASE}/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
     f"{BASE}/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"],
    MODEL_DIR / "hand_landmarker.task")

FACE_MODEL_PATH = download_with_fallback(
    [f"{BASE}/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
     f"{BASE}/face_landmarker/face_landmarker/float16/1/face_landmarker.task"],
    MODEL_DIR / "face_landmarker.task")

GESTURE_MODEL_PATH = download_with_fallback(
    [f"{BASE}/gesture_recognizer/gesture_recognizer/float16/latest/gesture_recognizer.task",
     f"{BASE}/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"],
    MODEL_DIR / "gesture_recognizer.task")

OBJECT_MODEL_PATH = download_with_fallback(
    [f"{BASE}/object_detector/efficientdet_lite0/float32/latest/efficientdet_lite0.tflite",
     f"{BASE}/object_detector/efficientdet_lite0/int8/latest/efficientdet_lite0.tflite",
     f"{BASE}/object_detector/efficientdet_lite0/float32/1/efficientdet_lite0.tflite"],
    MODEL_DIR / "efficientdet_lite0.tflite")

SETUP_SECONDS = _time.time() - SETUP_START
print(f"\nSetup complete ({SETUP_SECONDS:.0f}s)")


In [ ]:
# =====================================================================
# [2] Configuration + video input + shared utilities
#     This is the only cell most people need to edit. Everything downstream
#     reads from CONFIG below.
# =====================================================================
import json
import math
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------- Input video -------------------------
# Leave USE_GOOGLE_DRIVE = False to upload a file straight from your machine.
# Set it to True (and edit VIDEO_PATH) to read a video from Google Drive.
USE_GOOGLE_DRIVE = False
VIDEO_PATH = "/content/drive/MyDrive/your_video.mp4"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if not Path(VIDEO_PATH).exists():
    print(f"Video not found: {VIDEO_PATH}")
    print("Upload an MP4 to run the demo "
          "(10-30s, one person, full body + hands + a graspable object).")
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = "/content/" + next(iter(uploaded))
    print("Upload complete:", VIDEO_PATH)

# ------------------------- CONFIG -------------------------
CONFIG = {
    "video": {
        "analysis_fps": 15.0,         # analysis rate (source video is subsampled)
        "max_analysis_seconds": 40.0, # cap on analysed seconds (None = whole video)
        # Write one downscaled JPEG per analysed frame into timeline/frames/.
        # This is what turns each row of frames.jsonl into a complete
        # (vision + language + motion + phase) tuple usable as training data.
        "export_frame_images": True,
        "frame_image_width": 320,
        "output_width": 640,          # width of the overlay video
    },
    "mediapipe": {
        "use_gpu": True,              # prefer the GPU delegate, fall back to CPU
        "pose": True,
        "hands": True,
        "face": True,
        "gesture": True,
        "object_detection": True,
        "num_hands": 2,
        "mirror_handedness": False,   # True for mirrored selfie video (swaps L/R)
    },
    "object": {
        "target_labels": ["bottle", "cup", "bowl", "box", "ball",
                          "sports ball", "book", "cell phone"],
        # Alias table applied before tracking, so a wobbling detector label
        # does not split one object into several tracks.
        # Example: a glass flickers between cup / bowl / wine glass -> cup.
        "label_aliases": {"wine glass": "cup"},
        # Environment objects are tracked and recorded, but never become the
        # primary target of an interaction candidate. Tables are huge, so
        # leaving them in the target pool would hijack the whole analysis.
        "environment_labels": ["dining table", "chair", "couch", "bench", "bed"],
        "min_confidence": 0.35,
        "max_results": 5,
    },
    "behavior": {
        "task": "pick_and_place",     # task label written to behavior_summary.json
        "enable_interaction_candidates": True,
        "contact_distance": 0.30,     # hand-object contact threshold (normalised by object size)
        "reach_window": 5,            # frame window used for the reach candidate
        "moving_speed": 0.06,         # object "moving" threshold [normalised units/s]
    },
    "cleaning": {
        "max_interpolation_gap_frames": 3,
        "min_pose_confidence": 0.5,
        "min_hand_confidence": 0.5,
        "min_object_confidence": 0.35,
        "smoothing_alpha": 0.55,      # EMA factor (1.0 = no smoothing)
    },
    "mujoco": {
        "render_width": 960,
        "render_height": 720,
        "use_image_translation": True,  # map on-screen translation onto the root body
        "frame_world_width": 1.6,       # how many metres the frame width represents
    },
    "vrm": {
        "export_vrma": True,
        "export_fingers": True,
    },
}

# ------------------------- Project directories -------------------------
PROJECT_DIR = Path("/content/human_behavior_demo_2_0")
OUTPUT_DIR = PROJECT_DIR / "output"
OVERLAY_DIR = OUTPUT_DIR / "01_mediapipe_overlay"
MUJOCO_DIR = OUTPUT_DIR / "02_mujoco"
UNITY_DIR = OUTPUT_DIR / "03_unity_vrm"
DATASET_DIR = OUTPUT_DIR / "04_behavior_dataset"
COMPARISON_DIR = OUTPUT_DIR / "05_comparison"
SOURCE_DIR = PROJECT_DIR / "source"
WORK_DIR = PROJECT_DIR / "_work"

DATASET_SUBDIRS = ["human", "objects", "interactions", "timeline",
                   "metrics", "quality", "adapters"]

for directory in [PROJECT_DIR, OUTPUT_DIR, OVERLAY_DIR, MUJOCO_DIR, UNITY_DIR,
                  DATASET_DIR, COMPARISON_DIR, SOURCE_DIR, WORK_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
for name in DATASET_SUBDIRS:
    (DATASET_DIR / name).mkdir(parents=True, exist_ok=True)

# Persist the config next to the outputs, so a run is reproducible
(PROJECT_DIR / "config.json").write_text(
    json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")

# Keep a copy of the source video with the outputs
SOURCE_VIDEO = SOURCE_DIR / "source_video.mp4"
if not SOURCE_VIDEO.exists():
    shutil.copy(VIDEO_PATH, SOURCE_VIDEO)

# ------------------------- Probe the source video -------------------------
_cap = cv2.VideoCapture(str(SOURCE_VIDEO))
if not _cap.isOpened():
    raise RuntimeError(f"Could not open the video: {SOURCE_VIDEO}")
SOURCE_FPS = _cap.get(cv2.CAP_PROP_FPS)
SOURCE_FRAME_COUNT = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
SOURCE_WIDTH = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
_cap.release()
if SOURCE_FPS <= 0:
    raise RuntimeError("Could not read the video frame rate")
SOURCE_DURATION = SOURCE_FRAME_COUNT / SOURCE_FPS

# ------------------------- Landmark constants -------------------------
HAND_LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_FINGER_MCP", "INDEX_FINGER_PIP", "INDEX_FINGER_DIP", "INDEX_FINGER_TIP",
    "MIDDLE_FINGER_MCP", "MIDDLE_FINGER_PIP", "MIDDLE_FINGER_DIP", "MIDDLE_FINGER_TIP",
    "RING_FINGER_MCP", "RING_FINGER_PIP", "RING_FINGER_DIP", "RING_FINGER_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]
POSE_LANDMARK_NAMES = [
    "NOSE",
    "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER",
    "LEFT_EAR", "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT",
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW",
    "LEFT_WRIST", "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY",
    "LEFT_INDEX", "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB",
    "LEFT_HIP", "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE",
    "LEFT_ANKLE", "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]
HAND_BONES = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20), (0, 17),
]
POSE_BONES = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (24, 26), (26, 28), (27, 31), (28, 32),
]
FINGER_CHAINS = {
    "thumb": [0, 1, 2, 3, 4],
    "index": [0, 5, 6, 7, 8],
    "middle": [0, 9, 10, 11, 12],
    "ring": [0, 13, 14, 15, 16],
    "little": [0, 17, 18, 19, 20],
}
FINGER_ORDER = ["thumb", "index", "middle", "ring", "little"]


# ------------------------- Shared utilities -------------------------
def encode_h264(source_path, destination_path, fps):
    """Transcode to H.264 with ffmpeg so the video plays in the browser."""
    command = ["ffmpeg", "-y", "-loglevel", "error", "-r", str(fps),
               "-i", str(source_path), "-c:v", "libx264",
               "-pix_fmt", "yuv420p", "-movflags", "+faststart",
               str(destination_path)]
    result = subprocess.run(command, capture_output=True)
    if result.returncode != 0 or not Path(destination_path).exists():
        shutil.copy(source_path, destination_path)
    return str(destination_path)


def show_video(video_path, width=520):
    """Embed a video player in the notebook (skipped above 40 MB)."""
    import base64
    from IPython.display import HTML, display
    data = Path(video_path).read_bytes()
    if len(data) > 40 * 1024 * 1024:
        print(f"Video too large to embed, skipping preview: {video_path} "
              f"({len(data) / 1024 / 1024:.1f} MB)")
        return
    encoded = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls loop playsinline '
                 f'src="data:video/mp4;base64,{encoded}"></video>'))


def make_video_writer(path, fps, size):
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, size)
    if not writer.isOpened():
        raise RuntimeError(f"Could not create the output video: {path}")
    return writer


def even(value):
    return int(value) - int(value) % 2


print("Configuration ready")
print(f"input  : {SOURCE_VIDEO}")
print(f"          {SOURCE_WIDTH}x{SOURCE_HEIGHT}  {SOURCE_FPS:.2f}fps  "
      f"{SOURCE_FRAME_COUNT}frames  {SOURCE_DURATION:.1f}s")
print(f"output : {OUTPUT_DIR}")


In [ ]:
# =====================================================================
# [3] MediaPipe unified capture
#     Pose / Hands / Face / Gesture / Object are analysed in a single pass
#     and share one timestamp -- that shared clock is what later lets vision,
#     language, motion and phase line up on the same row of frames.jsonl.
#     GPU delegate is preferred; any task that fails falls back to CPU.
# =====================================================================
import time

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode

MP_CFG = CONFIG["mediapipe"]
ANALYSIS_FPS = CONFIG["video"]["analysis_fps"]
MAX_ANALYSIS_SECONDS = CONFIG["video"]["max_analysis_seconds"]

FRAME_STEP = max(1, int(round(SOURCE_FPS / ANALYSIS_FPS)))
EFFECTIVE_FPS = SOURCE_FPS / FRAME_STEP
MAX_SOURCE_FRAME = (int(MAX_ANALYSIS_SECONDS * SOURCE_FPS)
                    if MAX_ANALYSIS_SECONDS else SOURCE_FRAME_COUNT)

print(f"Analysis: keeping 1 of every {FRAME_STEP} frames -> {EFFECTIVE_FPS:.2f}fps")

# --------------------------- Task creation (GPU -> CPU fallback) ---------------------------
TASK_DELEGATE_LOG = {}


def create_task(task_name, factory):
    """Create a task on the GPU delegate, falling back to CPU on failure."""
    order = (["GPU", "CPU"] if (MP_CFG["use_gpu"] and HAS_GPU) else ["CPU"])
    requested = order[0]
    for delegate_name in order:
        try:
            delegate = (BaseOptions.Delegate.GPU if delegate_name == "GPU"
                        else BaseOptions.Delegate.CPU)
            task = factory(delegate)
            TASK_DELEGATE_LOG[task_name] = {
                "requested_delegate": requested,
                "actual_delegate": delegate_name,
            }
            print(f"  {task_name:<18} delegate={delegate_name}")
            return task
        except Exception as error:  # noqa: BLE001
            print(f"  {task_name:<18} {delegate_name} failed -> falling back ({error})")
    TASK_DELEGATE_LOG[task_name] = {"requested_delegate": requested,
                                    "actual_delegate": "unavailable"}
    return None


print("--- Initialising MediaPipe tasks ---")
pose_landmarker = None
if MP_CFG["pose"] and POSE_MODEL_PATH:
    pose_landmarker = create_task("pose", lambda d: (
        mp.tasks.vision.PoseLandmarker.create_from_options(
            mp.tasks.vision.PoseLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH, delegate=d),
                running_mode=VisionRunningMode.VIDEO,
                num_poses=1, output_segmentation_masks=False))))

hand_landmarker = None
if MP_CFG["hands"] and HAND_MODEL_PATH:
    hand_landmarker = create_task("hands", lambda d: (
        mp.tasks.vision.HandLandmarker.create_from_options(
            mp.tasks.vision.HandLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH, delegate=d),
                running_mode=VisionRunningMode.VIDEO,
                num_hands=MP_CFG["num_hands"]))))

face_landmarker = None
if MP_CFG["face"] and FACE_MODEL_PATH:
    face_landmarker = create_task("face", lambda d: (
        mp.tasks.vision.FaceLandmarker.create_from_options(
            mp.tasks.vision.FaceLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=FACE_MODEL_PATH, delegate=d),
                running_mode=VisionRunningMode.VIDEO,
                num_faces=1,
                output_face_blendshapes=True,
                output_facial_transformation_matrixes=True))))

gesture_recognizer = None
if MP_CFG["gesture"] and GESTURE_MODEL_PATH:
    gesture_recognizer = create_task("gesture", lambda d: (
        mp.tasks.vision.GestureRecognizer.create_from_options(
            mp.tasks.vision.GestureRecognizerOptions(
                base_options=BaseOptions(model_asset_path=GESTURE_MODEL_PATH, delegate=d),
                running_mode=VisionRunningMode.VIDEO,
                num_hands=MP_CFG["num_hands"]))))

object_detector = None
if MP_CFG["object_detection"] and OBJECT_MODEL_PATH:
    object_detector = create_task("object", lambda d: (
        mp.tasks.vision.ObjectDetector.create_from_options(
            mp.tasks.vision.ObjectDetectorOptions(
                base_options=BaseOptions(model_asset_path=OBJECT_MODEL_PATH, delegate=d),
                running_mode=VisionRunningMode.VIDEO,
                score_threshold=CONFIG["object"]["min_confidence"],
                max_results=CONFIG["object"]["max_results"]))))


def landmarks_to_array(landmark_list):
    return np.array([[lm.x, lm.y, lm.z] for lm in landmark_list], dtype=np.float32)


def landmarks_to_visibility(landmark_list):
    return np.array(
        [[float(getattr(lm, "visibility", 0.0) or 0.0),
          float(getattr(lm, "presence", 0.0) or 0.0)] for lm in landmark_list],
        dtype=np.float32)


# --------------------------- Analysis loop (one shared timestamp) ---------------------------
frames = []
source_frame_index = 0
last_timestamp_ms = -1
capture_start = time.time()

video = cv2.VideoCapture(str(SOURCE_VIDEO))

while True:
    success, frame_bgr = video.read()
    if not success:
        break
    if source_frame_index > MAX_SOURCE_FRAME:
        print(f"Stopped at max_analysis_seconds={MAX_ANALYSIS_SECONDS}s")
        break
    if source_frame_index % FRAME_STEP != 0:
        source_frame_index += 1
        continue

    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # Every task receives the same timestamp -- never let them drift apart
    timestamp_ms = max(int(round(source_frame_index * 1000 / SOURCE_FPS)),
                       last_timestamp_ms + 1)
    last_timestamp_ms = timestamp_ms

    record = {
        "frame_index": len(frames),
        "source_frame_index": source_frame_index,
        "timestamp_ms": timestamp_ms,
        "timestamp_sec": timestamp_ms / 1000.0,
        "pose": None, "hands": {}, "face": None,
        "gestures": {}, "objects": [],
    }

    # ---- Pose ----
    if pose_landmarker is not None:
        result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
        if result.pose_landmarks:
            record["pose"] = {
                "normalized": landmarks_to_array(result.pose_landmarks[0]),
                "visibility": landmarks_to_visibility(result.pose_landmarks[0]),
                "world": (landmarks_to_array(result.pose_world_landmarks[0])
                          if result.pose_world_landmarks else None),
            }

    # ---- Hands ----
    if hand_landmarker is not None:
        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)
        for hand_index, landmarks in enumerate(result.hand_landmarks):
            category = result.handedness[hand_index][0]
            label = category.category_name
            if MP_CFG["mirror_handedness"]:
                label = "Left" if label == "Right" else "Right"
            if label in record["hands"] and \
                    record["hands"][label]["score"] >= category.score:
                continue
            record["hands"][label] = {
                "score": float(category.score),
                "normalized": landmarks_to_array(landmarks),
                "world": landmarks_to_array(result.hand_world_landmarks[hand_index]),
            }

    # ---- Face ----
    if face_landmarker is not None:
        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
        if result.face_landmarks:
            blendshapes = {}
            if result.face_blendshapes:
                blendshapes = {c.category_name: float(c.score)
                               for c in result.face_blendshapes[0]}
            transform = None
            if result.facial_transformation_matrixes:
                transform = np.array(result.facial_transformation_matrixes[0],
                                     dtype=np.float32)
            record["face"] = {
                "landmarks": landmarks_to_array(result.face_landmarks[0]),
                "blendshapes": blendshapes,
                "transform": transform,
            }

    # ---- Gesture ----
    if gesture_recognizer is not None:
        result = gesture_recognizer.recognize_for_video(mp_image, timestamp_ms)
        for hand_index, gesture_list in enumerate(result.gestures):
            if not gesture_list:
                continue
            label = result.handedness[hand_index][0].category_name
            if MP_CFG["mirror_handedness"]:
                label = "Left" if label == "Right" else "Right"
            top = gesture_list[0]
            record["gestures"][label] = {
                "gesture": top.category_name,
                "score": float(top.score),
            }

    # ---- Object ----
    if object_detector is not None:
        result = object_detector.detect_for_video(mp_image, timestamp_ms)
        for detection in result.detections:
            box = detection.bounding_box
            category = detection.categories[0]
            record["objects"].append({
                "label": category.category_name,
                "score": float(category.score),
                "x_min": box.origin_x / SOURCE_WIDTH,
                "y_min": box.origin_y / SOURCE_HEIGHT,
                "width": box.width / SOURCE_WIDTH,
                "height": box.height / SOURCE_HEIGHT,
            })

    frames.append(record)
    source_frame_index += 1

    if len(frames) % 50 == 0:
        print(f"  {len(frames)} frames analysed "
              f"({time.time() - capture_start:.0f}s)")

video.release()
for task in [pose_landmarker, hand_landmarker, face_landmarker,
             gesture_recognizer, object_detector]:
    if task is not None:
        task.close()

CAPTURE_SECONDS = time.time() - capture_start
NUM_FRAMES = len(frames)
if NUM_FRAMES == 0:
    raise RuntimeError("No frames could be analysed -- please check the video.")

# --------------------------- Coverage + runtime_info.json ---------------------------
COVERAGE = {
    "total_frames": NUM_FRAMES,
    "pose": sum(1 for f in frames if f["pose"] is not None),
    "left_hand": sum(1 for f in frames if "Left" in f["hands"]),
    "right_hand": sum(1 for f in frames if "Right" in f["hands"]),
    "face": sum(1 for f in frames if f["face"] is not None),
    "gesture": sum(1 for f in frames if f["gestures"]),
    "objects": sum(1 for f in frames if f["objects"]),
}
FACE_LANDMARK_COUNT = next(
    (len(f["face"]["landmarks"]) for f in frames if f["face"] is not None), 0)

RUNTIME_INFO = {
    "gpu_name": GPU_NAME,
    "mujoco_gl": os.environ.get("MUJOCO_GL"),
    "tasks": TASK_DELEGATE_LOG,
    "processing_time_sec": round(CAPTURE_SECONDS, 1),
    "average_fps": round(NUM_FRAMES / CAPTURE_SECONDS, 2) if CAPTURE_SECONDS else None,
    "analysis_fps": EFFECTIVE_FPS,
    "frame_count": NUM_FRAMES,
}
(DATASET_DIR / "runtime_info.json").write_text(
    json.dumps(RUNTIME_INFO, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\nCapture complete ({CAPTURE_SECONDS:.1f}s / {NUM_FRAMES} frames / "
      f"{RUNTIME_INFO['average_fps']}fps)")
print(json.dumps(COVERAGE, indent=2))
print("Detected objects:",
      sorted({o['label'] for f in frames for o in f['objects']}) or "none")


In [ ]:
# =====================================================================
# [3.5] AI-assisted class re-mapping (optional cell)
#     Off-the-shelf detectors wobble: a glass may come back as "bowl" in one
#     frame and "wine glass" in the next, which splits it into several tracks.
#     Here we show crops of each detected object to Gemini, ask it to rank the
#     best COCO classes, let a human confirm, and feed the result back into
#     [4] as label_aliases.
#     The raw detector labels are never overwritten -- they stay in
#     objects/object_detections.csv so provenance is preserved.
# =====================================================================
import base64
import sys
import urllib.request


def ask(prompt="> "):
    """input() with a SHORT prompt.

    Colab draws the whole prompt string on one line and puts the text box
    right after it, so a long prompt pushes the box off screen (a trailing
    "\n" is not honoured either). Print the guidance with print() and keep
    the prompt itself down to a couple of characters.
    Falls back to non-interactive mode when stdin is unavailable
    (Run all / nbconvert / papermill).
    """
    global RECLASSIFY_INTERACTIVE
    sys.stdout.flush()
    try:
        return input(prompt).strip().lower()
    except Exception as error:  # EOFError / StdinNotImplementedError
        print(f"  stdin is unavailable ({type(error).__name__}) -> "
              "continuing without prompts")
        RECLASSIFY_INTERACTIVE = False
        return ""


RECLASSIFY_INTERACTIVE = True   # False: accept Gemini's top pick without prompting
RECLASSIFY_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite",
                     "gemini-2.5-flash-lite", "flash-lite-latest"]
RECLASSIFY_CROPS_PER_LABEL = 3
# True: discard the previous run's label_aliases and start over.
# False appends to them (only useful when tuning the same video repeatedly).
RECLASSIFY_RESET_PREVIOUS = True
# Classes that can never be a tracking target. Keeps a close-up of a hand
# from being re-labelled "person".
RECLASSIFY_EXCLUDE_FROM_CANDIDATES = {"person"}
# True makes every detected label configurable (including person / rare labels)
RECLASSIFY_INCLUDE_ALL = False
RECLASSIFY_SKIP_LABELS = {"person"}   # never re-classified; ignored if INCLUDE_ALL
RECLASSIFY_MIN_COUNT = 1              # labels seen fewer times than this are skipped

# The 80-class COCO vocabulary used by EfficientDet-Lite
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
    "truck", "boat", "traffic light", "fire hydrant", "stop sign",
    "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork",
    "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush"]

# Vocabulary offered to the model (minus classes that can never be tracked)
CANDIDATE_CLASSES = [c for c in COCO_CLASSES
                     if c not in RECLASSIFY_EXCLUDE_FROM_CANDIDATES]

# Drop aliases from a previous run so that swapping the video mid-session
# does not silently carry the old video's mapping over.
if RECLASSIFY_RESET_PREVIOUS:
    _previous = CONFIG["object"].get("label_aliases", {})
    if _previous:
        print(f"Discarding previous label_aliases: {_previous}")
    CONFIG["object"]["label_aliases"] = {}

# ---------------- Collect representative crops per detected label ----------------
all_label_samples = {}   # every detected label -> [(score, frame_index, det)]
for record in frames:
    for det in record["objects"]:
        all_label_samples.setdefault(det["label"], []).append(
            (det["score"], record["frame_index"], det))

# Split into configurable / excluded. Excluded labels keep a reason so the
# final table can explain every label the detector produced.
min_count = 1 if RECLASSIFY_INCLUDE_ALL else RECLASSIFY_MIN_COUNT
skip_labels = set() if RECLASSIFY_INCLUDE_ALL else RECLASSIFY_SKIP_LABELS
label_samples, EXCLUDED_LABELS = {}, {}
for label, samples in all_label_samples.items():
    if label in skip_labels:
        EXCLUDED_LABELS[label] = (f"skip_label ({len(samples)} detections / "
                                  "RECLASSIFY_SKIP_LABELS)")
    elif len(samples) < min_count:
        EXCLUDED_LABELS[label] = (f"low_count ({len(samples)} < "
                                  f"RECLASSIFY_MIN_COUNT={min_count})")
    else:
        label_samples[label] = samples

print(f"Detected labels: {len(all_label_samples)} -> configurable: {len(label_samples)}")
if EXCLUDED_LABELS:
    print("Excluded (set RECLASSIFY_INCLUDE_ALL = True to configure them all):")
    for label, reason in EXCLUDED_LABELS.items():
        print(f"  - {label}: {reason}")

if not label_samples:
    print("No label to re-classify (person only, or nothing detected). Skipping.")
else:
    _key = None
    try:
        from google.colab import userdata
        _key = userdata.get("GEMINI_API_KEY")
    except Exception:
        _key = os.environ.get("GEMINI_API_KEY")

    video = cv2.VideoCapture(str(SOURCE_VIDEO))
    label_crops = {}   # label -> [jpeg bytes]
    for label, samples in label_samples.items():
        # Take the highest-scoring detections, then spread them over time
        samples = sorted(samples, key=lambda s: -s[0])[:12]
        samples = sorted(samples, key=lambda s: s[1])
        step = max(1, len(samples) // RECLASSIFY_CROPS_PER_LABEL)
        crops = []
        for score, fi, det in samples[::step][:RECLASSIFY_CROPS_PER_LABEL]:
            video.set(cv2.CAP_PROP_POS_FRAMES, frames[fi]["source_frame_index"])
            success, image = video.read()
            if not success:
                continue
            ih, iw = image.shape[:2]
            mx, my = det["width"] * 0.25, det["height"] * 0.25
            x0 = int(max(0.0, det["x_min"] - mx) * iw)
            y0 = int(max(0.0, det["y_min"] - my) * ih)
            x1 = int(min(1.0, det["x_min"] + det["width"] + mx) * iw)
            y1 = int(min(1.0, det["y_min"] + det["height"] + my) * ih)
            crop = image[y0:y1, x0:x1]
            if crop.size == 0:
                continue
            if crop.shape[1] > 320:
                crop = cv2.resize(crop, (320, int(crop.shape[0] * 320 / crop.shape[1])))
            ok, buffer = cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 85])
            if ok:
                crops.append(buffer.tobytes())
        if crops:
            label_crops[label] = crops
    video.release()

    if not label_crops:
        print("Could not read any crop out of the video -- nothing to "
              "confirm. Re-run [3] and check that the source video is "
              "readable.")

    # ---------------- Ask Gemini for the best class out of the 80 ----------------
    proposals = {}   # detected_label -> [{"class", "reason"}, ...] best first
    if _key and label_crops:
        # Blind judgement: the detector's own label is never sent. Groups are
        # anonymous ids, so the model cannot anchor on the wrong guess. The
        # mapping back to labels stays local.
        group_ids = {label: f"group_{i + 1}"
                     for i, label in enumerate(label_crops)}
        parts = [{"text": (
            "You will see cropped images of objects from a video. Images "
            "with the same group id show the same physical object. For EACH "
            "group, rank up to 5 candidate classes from the following fixed "
            "COCO vocabulary, from most to least suitable for the "
            "object in the crops. Judge the object itself, not the person or "
            "body parts that may appear around it. "
            "You MUST choose from this list only:\n"
            + ", ".join(CANDIDATE_CLASSES) + "\n"
            "Judge purely from what is visible. Respond ONLY with a JSON "
            'array: [{"group_id": "group_1", "candidates": '
            '[{"rank": 1, "class": "cup", "reason": "short reason in English"}, '
            '{"rank": 2, "class": "bowl", "reason": "..."}]}]')}]
        for label, crops in label_crops.items():
            parts.append({"text": f"Group id: {group_ids[label]}"})
            for jpeg in crops:
                parts.append({"inline_data": {
                    "mime_type": "image/jpeg",
                    "data": base64.b64encode(jpeg).decode("ascii")}})
        body = json.dumps({
            "contents": [{"parts": parts}],
            "generationConfig": {"temperature": 0.1,
                                 "response_mime_type": "application/json"},
        }).encode("utf-8")
        for model_name in RECLASSIFY_MODELS:
            request = urllib.request.Request(
                f"https://generativelanguage.googleapis.com/v1beta/models/"
                f"{model_name}:generateContent",
                data=body, method="POST",
                headers={"Content-Type": "application/json",
                         "x-goog-api-key": _key})
            try:
                with urllib.request.urlopen(request, timeout=120) as res:
                    payload = json.loads(res.read().decode("utf-8"))
                text = "".join(p.get("text", "") for p in
                               payload["candidates"][0]["content"]["parts"]).strip()
                if text.startswith("```"):
                    text = text.split("```")[1]
                    if text.startswith("json"):
                        text = text[4:]
                id_to_label = {gid: label for label, gid in group_ids.items()}
                for item in json.loads(text):
                    seen, candidates = set(), []
                    for cand in item.get("candidates", []):
                        cls = str(cand.get("class", "")).strip().lower()
                        if cls in CANDIDATE_CLASSES and cls not in seen:
                            seen.add(cls)
                            candidates.append({
                                "class": cls,
                                "reason": str(cand.get("reason", "")).strip()})
                        if len(candidates) >= 5:
                            break
                    label_key = id_to_label.get(item.get("group_id"))
                    if candidates and label_key:   # ignore unknown groups
                        proposals[label_key] = candidates
                print(f"Received Gemini judgement (model={model_name})")
                break
            except Exception as error:
                print(f"  {model_name} failed -> trying next model ({error})")
    elif not _key:
        print("GEMINI_API_KEY is not set, skipping AI suggestions "
              "(manual re-classification still works).")

    # ---------------- Human confirmation -> reapply as label_aliases ----------------
    RECLASSIFICATIONS = []
    aliases = CONFIG["object"].setdefault("label_aliases", {})
    for label in label_crops:
        # For each label: show crops -> confirm class -> confirm role
        crops = label_crops[label]
        try:
            from IPython.display import display, Image as IPImage, HTML
            _env = CONFIG["object"].get("environment_labels", [])
            _role_now = "environment" if label in _env else "target"
            display(HTML(f"Crops for detected label <b><code>{label}</code></b>"
                         f" (current role: {_role_now}):"))
            for jpeg in crops:
                display(IPImage(data=jpeg, height=140))
        except Exception:
            pass

        candidates = proposals.get(label, [])
        if candidates:
            print(f"\nGemini candidates for '{label}' (best first):")
            for rank, cand in enumerate(candidates, start=1):
                mark = " (same as detector)" if cand["class"] == label else ""
                print(f"  {rank}. {cand['class']}{mark} - {cand['reason']}")
        top = candidates[0]["class"] if candidates else None
        if RECLASSIFY_INTERACTIVE:
            hint = f"Enter = accept top pick [{top}]" if top else "Enter = keep"
            print(f"\n'{label}': {hint} / 1-{max(len(candidates), 1)} = pick "
                  f"candidate / k = keep / or type a COCO class")
            answer = ask("class> ")
            if answer == "" and top:
                final, source, rank = top, "gemini_confirmed", 1
            elif answer in ("", "k"):
                final, source, rank = label, "detector", None
            elif answer.isdigit() and 1 <= int(answer) <= len(candidates):
                rank = int(answer)
                final = candidates[rank - 1]["class"]
                source = "gemini_confirmed" if rank == 1 else "gemini_ranked"
            elif answer in CANDIDATE_CLASSES:
                final, source, rank = answer, "manual", None
            else:
                print(f"  '{answer}' is not in the vocabulary. "
                      f"Keeping '{label}'.")
                final, source, rank = label, "detector", None
        else:
            final = top or label
            source = "gemini_auto" if top else "detector"
            rank = 1 if top else None
        # ---- Confirm the role: manipulation target vs environment ----
        env_labels = CONFIG["object"].setdefault("environment_labels", [])
        # Default role: anything already in environment_labels, plus labels
        # that can never be manipulated, defaults to "environment".
        # Making a table a target would hijack the primary object and break
        # every interaction candidate downstream.
        current_role = ("environment"
                        if (final in env_labels
                            or final in RECLASSIFY_SKIP_LABELS)
                        else "target")
        role_names = {"target": "target (manipulated)",
                      "environment": "environment (scene)"}
        if RECLASSIFY_INTERACTIVE:
            print(f"  role of '{final}': currently "
                  f"[{role_names[current_role]}] -> "
                  f"Enter = keep / t = target / e = environment")
            role_answer = ask("role> ")
            role = {"t": "target", "e": "environment"}.get(
                role_answer, current_role)
        else:
            role = current_role
        if role == "environment":
            if final not in env_labels:
                env_labels.append(final)
        else:
            if final in env_labels:
                env_labels.remove(final)
            if final not in CONFIG["object"]["target_labels"]:
                CONFIG["object"]["target_labels"].append(final)
        if final != label:
            aliases[label] = final
        RECLASSIFICATIONS.append({
            "detected_label": label,
            "gemini_candidates": candidates,
            "final_class": final, "chosen_rank": rank, "source": source,
            "role": role})
        print(f"  {label} -> {final} ({source}) / role: {role_names[role]}")

    # ---------------- Final mapping table for every detected label ----------------
    print("\n" + "=" * 78)
    print("Object label mapping")
    print("=" * 78)
    header = (f"{'detected':<18}{'count':>6}  {'-> final class':<18}"
              f"{'role':<14}{'basis'}")
    print(header)
    print("-" * 78)
    decided = {r["detected_label"]: r for r in RECLASSIFICATIONS}
    role_label = {"target": "target", "environment": "environment"}
    LABEL_TABLE = []
    for label, samples in sorted(all_label_samples.items(),
                                 key=lambda kv: -len(kv[1])):
        record = decided.get(label)
        if record:
            final, role = record["final_class"], role_label[record["role"]]
            source, excluded = record["source"], None
        else:
            final, role = "-", "excluded"
            source = EXCLUDED_LABELS.get(label, "excluded")
            excluded = EXCLUDED_LABELS.get(label)
        mark = "" if final == label or final == "-" else " *"
        print(f"{label:<18}{len(samples):>6}  {final + mark:<18}"
              f"{role:<14}{source}")
        LABEL_TABLE.append({
            "detected_label": label, "detection_count": len(samples),
            "final_class": None if final == "-" else final,
            "role": record["role"] if record else None,
            "source": source, "excluded_reason": excluded,
            "changed": bool(record and record["final_class"] != label),
        })
    print("-" * 78)
    print("* = changed from the raw detector label")
    print(f"label_aliases to apply: {aliases or '(none)'}")
    print(f"target objects      : "
          f"{[r['detected_label'] for r in RECLASSIFICATIONS if r['role'] == 'target'] or 'none'}")
    print(f"environment objects : "
          f"{[r['detected_label'] for r in RECLASSIFICATIONS if r['role'] == 'environment'] or 'none'}")
    print("-> Run [4] next and the re-classification is applied to tracking")

    (DATASET_DIR / "objects").mkdir(exist_ok=True)
    (DATASET_DIR / "objects/label_reclassification.json").write_text(
        json.dumps({"note": ("Mapping from raw detector labels to the "
                             "human-confirmed best COCO class. Applied as "
                             "label_aliases before tracking. Raw labels are "
                             "preserved in objects/object_detections.csv."),
                    # The detector's label is never revealed to the model
                    "gemini_blind_judgment": True,

                    "coco_vocabulary_size": len(COCO_CLASSES),
                    "reclassifications": RECLASSIFICATIONS,
                    "label_table": LABEL_TABLE,
                    "excluded_labels": EXCLUDED_LABELS,
                    "resulting_label_aliases": aliases},
                   indent=2, ensure_ascii=False), encoding="utf-8")
    (PROJECT_DIR / "config.json").write_text(
        json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:
# =====================================================================
# [4] Build the Common Behavior Data
#     This is the heart of the demo. Object tracking, smoothing, bone
#     rotations, interaction candidates, metrics and quality are all computed
#     here into one engine-independent representation. MuJoCo and VRM later
#     read *from* this -- they never read from MediaPipe directly.
# =====================================================================
CLEAN = CONFIG["cleaning"]
BEH = CONFIG["behavior"]
ALPHA = CLEAN["smoothing_alpha"]
DT = 1.0 / EFFECTIVE_FPS

# =====================================================================
# 4-A. Object tracking (association by class + IoU + centre distance)
# =====================================================================
def bbox_iou(a, b):
    ax0, ay0, ax1, ay1 = a["x_min"], a["y_min"], a["x_min"] + a["width"], a["y_min"] + a["height"]
    bx0, by0, bx1, by1 = b["x_min"], b["y_min"], b["x_min"] + b["width"], b["y_min"] + b["height"]
    ix = max(0.0, min(ax1, bx1) - max(ax0, bx0))
    iy = max(0.0, min(ay1, by1) - max(ay0, by0))
    inter = ix * iy
    union = a["width"] * a["height"] + b["width"] * b["height"] - inter
    return inter / union if union > 0 else 0.0


TARGET_LABELS = set(CONFIG["object"]["target_labels"])
ENV_LABELS = set(CONFIG["object"].get("environment_labels", []))
TRACKED_LABELS = TARGET_LABELS | ENV_LABELS   # both roles are tracked
LABEL_ALIASES = CONFIG["object"].get("label_aliases", {})
TRACKS = []  # {track_id,label,obs:{frame_index:det},last_frame,last_center}

for record in frames:
    # Apply aliases before filtering, so a wobbling detector label does not
    # split one physical object into several tracks.
    detections = [dict(d, detector_label=d["label"],
                       label=LABEL_ALIASES.get(d["label"], d["label"]))
                  for d in record["objects"]]
    detections = [d for d in detections
                  if not TRACKED_LABELS or d["label"] in TRACKED_LABELS]
    used = set()
    for det in sorted(detections, key=lambda d: -d["score"]):
        center = np.array([det["x_min"] + det["width"] / 2,
                           det["y_min"] + det["height"] / 2])
        best, best_score = None, -1.0
        for track in TRACKS:
            if id(track) in used or track["label"] != det["label"]:
                continue
            if record["frame_index"] - track["last_frame"] > 12:
                continue
            iou = bbox_iou(det, track["obs"][track["last_frame"]])
            dist = float(np.linalg.norm(center - track["last_center"]))
            score = iou * 2.0 + max(0.0, 1.0 - dist / 0.25)
            if (iou > 0.05 or dist < 0.15) and score > best_score:
                best, best_score = track, score
        if best is None:
            best = {"track_id": f"obj_{len(TRACKS) + 1:03d}",
                    "label": det["label"], "obs": {},
                    "last_frame": record["frame_index"], "last_center": center}
            TRACKS.append(best)
        best["obs"][record["frame_index"]] = det
        best["last_frame"] = record["frame_index"]
        best["last_center"] = center
        used.add(id(best))

# Merge same-label tracks that never overlap in time.
# When a hand covers an object mid-lift the detection drops out and the same
# cup comes back as a new track. Two cups visible at once *do* overlap in
# time, so they are correctly left separate.
if CONFIG["object"].get("merge_same_label_tracks", True) and len(TRACKS) > 1:
    merged_tracks = []
    for track in sorted(TRACKS, key=lambda t: min(t["obs"])):
        target = None
        for cand in merged_tracks:
            if cand["label"] == track["label"] and \
                    not (set(cand["obs"]) & set(track["obs"])):
                target = cand
                break
        if target is None:
            merged_tracks.append(track)
        else:
            target["obs"].update(track["obs"])
    if len(merged_tracks) < len(TRACKS):
        print(f"Merged tracks: {len(TRACKS)} -> {len(merged_tracks)}")
    TRACKS = merged_tracks
    for i, track in enumerate(sorted(TRACKS, key=lambda t: min(t["obs"]))):
        track["track_id"] = f"obj_{i + 1:03d}"

# Build each track's time series (centre, velocity, moving/stationary)
for track in TRACKS:
    centers = np.full((NUM_FRAMES, 2), np.nan)
    sizes = np.full((NUM_FRAMES, 2), np.nan)
    scores = np.full(NUM_FRAMES, np.nan)
    for fi, det in track["obs"].items():
        centers[fi] = [det["x_min"] + det["width"] / 2,
                       det["y_min"] + det["height"] / 2]
        sizes[fi] = [det["width"], det["height"]]
        scores[fi] = det["score"]
    valid = ~np.isnan(centers[:, 0])
    idx = np.arange(NUM_FRAMES)
    if valid.sum() >= 2:  # interpolate short gaps only (max_interpolation_gap_frames)
        gap = CLEAN["max_interpolation_gap_frames"]
        filled = valid.copy()
        for fi in np.where(~valid)[0]:
            prev_valid = idx[valid & (idx < fi)]
            next_valid = idx[valid & (idx > fi)]
            if len(prev_valid) and len(next_valid) and \
                    next_valid[0] - prev_valid[-1] <= gap + 1:
                filled[fi] = True
        for column in range(2):
            centers[:, column] = np.interp(idx, idx[valid], centers[valid, column])
            sizes[:, column] = np.interp(idx, idx[valid], sizes[valid, column])
        track["interp_valid"] = filled
    else:
        track["interp_valid"] = valid
    velocity = np.zeros_like(centers)
    velocity[1:] = (centers[1:] - centers[:-1]) / DT
    speed = np.linalg.norm(velocity, axis=1)
    track["centers"], track["sizes"], track["scores"] = centers, sizes, scores
    track["velocity"], track["speed"], track["valid_raw"] = velocity, speed, valid
    track["moving"] = speed > BEH["moving_speed"]

# Assign roles: target (manipulated) vs environment (scene furniture)
for track in TRACKS:
    track["role"] = "environment" if track["label"] in ENV_LABELS else "target"

# The primary object is the target track seen in the most frames.
# Environment objects are excluded: they are large and detected in every
# frame, so they would always win and never be interacted with anyway.
_target_tracks = [t for t in TRACKS if t["role"] == "target"]
PRIMARY_TRACK = (max(_target_tracks, key=lambda t: len(t["obs"]))
                 if _target_tracks else None)
if PRIMARY_TRACK:
    print(f"Primary object: {PRIMARY_TRACK['track_id']} ({PRIMARY_TRACK['label']}, "
          f"{len(PRIMARY_TRACK['obs'])} frames)")
else:
    print("No target object detected -- behavior data will contain the human only")

# =====================================================================
# 4-B. Landmark conditioning (gap filling + EMA smoothing)
#      The raw landmarks are never overwritten -- derived series live
#      alongside them so consumers can always trace back to the observation.
# =====================================================================
def stack_series(getter, num_points):
    """Stack frames into (F, N, 3) plus a valid mask; fill gaps with the
    nearest valid frame."""
    array = np.full((NUM_FRAMES, num_points, 3), np.nan, dtype=np.float32)
    valid = np.zeros(NUM_FRAMES, dtype=bool)
    for fi, record in enumerate(frames):
        value = getter(record)
        if value is not None and len(value) == num_points:
            array[fi] = value
            valid[fi] = True
    if valid.any():
        valid_idx = np.where(valid)[0]
        for fi in range(NUM_FRAMES):
            if not valid[fi]:
                array[fi] = array[valid_idx[np.argmin(np.abs(valid_idx - fi))]]
    return array, valid


def ema_smooth(array, alpha):
    out = array.copy()
    for fi in range(1, len(out)):
        out[fi] = alpha * array[fi] + (1 - alpha) * out[fi - 1]
    return out


POSE_W_RAW, POSE_VALID = stack_series(
    lambda r: r["pose"]["world"] if r["pose"] else None, 33)
POSE_N_RAW, _ = stack_series(
    lambda r: r["pose"]["normalized"] if r["pose"] else None, 33)
HAND_W_RAW, HAND_VALID, HAND_N_RAW = {}, {}, {}
for hand_label in ["Left", "Right"]:
    HAND_W_RAW[hand_label], HAND_VALID[hand_label] = stack_series(
        lambda r, L=hand_label: r["hands"][L]["world"] if L in r["hands"] else None, 21)
    HAND_N_RAW[hand_label], _ = stack_series(
        lambda r, L=hand_label: r["hands"][L]["normalized"] if L in r["hands"] else None, 21)

if not POSE_VALID.any():
    raise RuntimeError("Pose was never detected. Use a video where the full body is visible.")

# Convert to the canonical frame.
# MediaPipe is x-right / y-down / z-into-screen; canonical CBD space is
# Y-up, right-handed, with the person facing +Z. Every adapter converts out
# of this one frame -- MuJoCo (Z-up) and glTF/VRM are pure transforms of it.
def to_canonical(points):
    out = points.copy()
    out[..., 1] *= -1.0
    out[..., 2] *= -1.0
    return out


POSE_C = ema_smooth(to_canonical(POSE_W_RAW), ALPHA)
HAND_C = {L: ema_smooth(to_canonical(HAND_W_RAW[L]), ALPHA) for L in ["Left", "Right"]}
POSE_IMG = ema_smooth(POSE_N_RAW.copy(), ALPHA)   # image space (root translation + distances)

# =====================================================================
# 4-C. Quaternion helpers + common bone rotations (T-pose reference)
# =====================================================================
def q_normalize(q):
    return q / (np.linalg.norm(q) + 1e-12)


def q_mul(a, b):
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,
                     aw*bx + ax*bw + ay*bz - az*by,
                     aw*by - ax*bz + ay*bw + az*bx,
                     aw*bz + ax*by - ay*bx + az*bw])


def q_conj(q):
    return np.array([q[0], -q[1], -q[2], -q[3]])


def normalize_vec(v, fallback):
    length = np.linalg.norm(v)
    return v / length if length > 1e-8 else np.array(fallback, dtype=float)


def quat_from_axes(x_axis, y_axis, z_axis):
    """Rotation matrix from column vectors x, y, z -> quaternion (w, x, y, z)."""
    m = np.column_stack([x_axis, y_axis, z_axis])
    tr = np.trace(m)
    if tr > 0:
        s = math.sqrt(tr + 1.0) * 2
        q = np.array([0.25*s, (m[2,1]-m[1,2])/s, (m[0,2]-m[2,0])/s, (m[1,0]-m[0,1])/s])
    elif m[0,0] > m[1,1] and m[0,0] > m[2,2]:
        s = math.sqrt(1.0 + m[0,0] - m[1,1] - m[2,2]) * 2
        q = np.array([(m[2,1]-m[1,2])/s, 0.25*s, (m[0,1]+m[1,0])/s, (m[0,2]+m[2,0])/s])
    elif m[1,1] > m[2,2]:
        s = math.sqrt(1.0 + m[1,1] - m[0,0] - m[2,2]) * 2
        q = np.array([(m[0,2]-m[2,0])/s, (m[0,1]+m[1,0])/s, 0.25*s, (m[1,2]+m[2,1])/s])
    else:
        s = math.sqrt(1.0 + m[2,2] - m[0,0] - m[1,1]) * 2
        q = np.array([(m[1,0]-m[0,1])/s, (m[0,2]+m[2,0])/s, (m[1,2]+m[2,1])/s, 0.25*s])
    return q_normalize(q)


def frame_quat(x_hint, y_hint):
    """Build an orthonormal frame quaternion from an x hint and a y hint."""
    y_axis = normalize_vec(y_hint, [0, 1, 0])
    x_axis = normalize_vec(x_hint - np.dot(x_hint, y_axis) * y_axis, [1, 0, 0])
    z_axis = np.cross(x_axis, y_axis)
    return quat_from_axes(x_axis, y_axis, z_axis)


def shortest_arc(a, b):
    a = normalize_vec(np.asarray(a, float), [0, 1, 0])
    b = normalize_vec(np.asarray(b, float), [0, 1, 0])
    d = float(np.dot(a, b))
    if d > 0.999999:
        return np.array([1.0, 0, 0, 0])
    if d < -0.999999:
        axis = np.cross(a, [1.0, 0, 0])
        if np.linalg.norm(axis) < 1e-6:
            axis = np.cross(a, [0, 1.0, 0])
        axis = normalize_vec(axis, [0, 0, 1])
        return np.array([0.0, axis[0], axis[1], axis[2]])
    axis = np.cross(a, b)
    q = np.array([1.0 + d, axis[0], axis[1], axis[2]])
    return q_normalize(q)


# Skeleton hierarchy and T-pose rest directions (canonical, Y-up, facing +Z)
BONE_PARENT = {
    "hips": None, "spine": "hips", "chest": "spine", "neck": "chest", "head": "neck",
    "left_shoulder": "chest", "left_upper_arm": "left_shoulder",
    "left_lower_arm": "left_upper_arm", "left_hand": "left_lower_arm",
    "right_shoulder": "chest", "right_upper_arm": "right_shoulder",
    "right_lower_arm": "right_upper_arm", "right_hand": "right_lower_arm",
    "left_upper_leg": "hips", "left_lower_leg": "left_upper_leg", "left_foot": "left_lower_leg",
    "right_upper_leg": "hips", "right_lower_leg": "right_upper_leg", "right_foot": "right_lower_leg",
}
BONE_ORDER = list(BONE_PARENT.keys())
REST_DIR = {
    "left_upper_arm": [1, 0, 0], "left_lower_arm": [1, 0, 0],
    "right_upper_arm": [-1, 0, 0], "right_lower_arm": [-1, 0, 0],
    "left_upper_leg": [0, -1, 0], "left_lower_leg": [0, -1, 0],
    "right_upper_leg": [0, -1, 0], "right_lower_leg": [0, -1, 0],
    "left_foot": [0, 0, 1], "right_foot": [0, 0, 1],
}
P = {name: i for i, name in enumerate(POSE_LANDMARK_NAMES)}


def hand_world_quat(points, side):
    """World orientation of a hand from its canonical landmarks.
    T-pose reference: fingers point +X (left) / -X (right), palm normal -Y."""
    fingers = normalize_vec(points[9] - points[0], [1 if side == "Left" else -1, 0, 0])
    raw_normal = np.cross(points[5] - points[0], points[17] - points[0])
    palm_normal = normalize_vec(-raw_normal if side == "Left" else raw_normal, [0, -1, 0])
    y_axis = normalize_vec(-palm_normal, [0, 1, 0])
    if side == "Left":
        x_axis = fingers
    else:
        x_axis = -fingers
    x_axis = normalize_vec(x_axis - np.dot(x_axis, y_axis) * y_axis, [1, 0, 0])
    z_axis = np.cross(x_axis, y_axis)
    return quat_from_axes(x_axis, y_axis, z_axis)


IDENTITY_Q = np.array([1.0, 0, 0, 0])
BONE_ROT = {bone: np.tile(IDENTITY_Q, (NUM_FRAMES, 1)) for bone in BONE_ORDER}
HIPS_POS_C = np.zeros((NUM_FRAMES, 3))
MJ = CONFIG["mujoco"]

for fi in range(NUM_FRAMES):
    pw = POSE_C[fi]
    hip_center = (pw[P["LEFT_HIP"]] + pw[P["RIGHT_HIP"]]) / 2
    shoulder_center = (pw[P["LEFT_SHOULDER"]] + pw[P["RIGHT_SHOULDER"]]) / 2
    up = shoulder_center - hip_center

    world = {}
    world["hips"] = frame_quat(pw[P["LEFT_HIP"]] - pw[P["RIGHT_HIP"]], up)
    chest_q = frame_quat(pw[P["LEFT_SHOULDER"]] - pw[P["RIGHT_SHOULDER"]], up)
    world["spine"] = chest_q
    world["chest"] = chest_q
    world["neck"] = chest_q
    ear_center = (pw[P["LEFT_EAR"]] + pw[P["RIGHT_EAR"]]) / 2
    head_forward = pw[P["NOSE"]] - ear_center
    head_lateral = pw[P["LEFT_EAR"]] - pw[P["RIGHT_EAR"]]
    head_up = np.cross(head_forward, head_lateral)
    world["head"] = frame_quat(head_lateral, head_up)
    world["left_shoulder"] = chest_q
    world["right_shoulder"] = chest_q

    limb_pairs = {
        "left_upper_arm": ("LEFT_SHOULDER", "LEFT_ELBOW"),
        "left_lower_arm": ("LEFT_ELBOW", "LEFT_WRIST"),
        "right_upper_arm": ("RIGHT_SHOULDER", "RIGHT_ELBOW"),
        "right_lower_arm": ("RIGHT_ELBOW", "RIGHT_WRIST"),
        "left_upper_leg": ("LEFT_HIP", "LEFT_KNEE"),
        "left_lower_leg": ("LEFT_KNEE", "LEFT_ANKLE"),
        "right_upper_leg": ("RIGHT_HIP", "RIGHT_KNEE"),
        "right_lower_leg": ("RIGHT_KNEE", "RIGHT_ANKLE"),
        "left_foot": ("LEFT_ANKLE", "LEFT_FOOT_INDEX"),
        "right_foot": ("RIGHT_ANKLE", "RIGHT_FOOT_INDEX"),
    }
    for bone, (a, b) in limb_pairs.items():
        world[bone] = q_mul(shortest_arc(REST_DIR[bone], pw[P[b]] - pw[P[a]]),
                            IDENTITY_Q)

    for side in ["Left", "Right"]:
        key = f"{side.lower()}_hand"
        if HAND_VALID[side].any():
            world[key] = hand_world_quat(HAND_C[side][fi], side)
        else:  # no hand detected at all -- fall back to the pose index finger
            wrist, index = pw[P[f"{side.upper()}_WRIST"]], pw[P[f"{side.upper()}_INDEX"]]
            world[key] = q_mul(shortest_arc(
                [1, 0, 0] if side == "Left" else [-1, 0, 0], index - wrist), IDENTITY_Q)

    # World orientation -> parent-relative local rotation (sign-continuous)
    for bone in BONE_ORDER:
        parent = BONE_PARENT[bone]
        local = (world[bone] if parent is None
                 else q_normalize(q_mul(q_conj(world[parent]), world[bone])))
        if fi > 0 and np.dot(local, BONE_ROT[bone][fi - 1]) < 0:
            local = -local
        BONE_ROT[bone][fi] = local

    # Root translation, derived from image-space hip motion
    if MJ["use_image_translation"]:
        hip_img = (POSE_IMG[fi][P["LEFT_HIP"]] + POSE_IMG[fi][P["RIGHT_HIP"]]) / 2
        HIPS_POS_C[fi] = [(hip_img[0] - 0.5) * MJ["frame_world_width"],
                          (0.5 - hip_img[1]) * MJ["frame_world_width"] * 0.5, 0.0]

HIPS_POS_C = ema_smooth(HIPS_POS_C, ALPHA)

# =====================================================================
# 4-D. Finger flexion angles + body joint angles
# =====================================================================
def angle_deg(v1, v2):
    c = np.dot(v1, v2) / ((np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-9)
    return math.degrees(math.acos(np.clip(c, -1.0, 1.0)))


FINGER_FLEX = {"Left": np.zeros((NUM_FRAMES, 5, 3)),
               "Right": np.zeros((NUM_FRAMES, 5, 3))}
for side in ["Left", "Right"]:
    if not HAND_VALID[side].any():
        continue
    for fi in range(NUM_FRAMES):
        hw = HAND_C[side][fi]
        for finger_i, finger in enumerate(FINGER_ORDER):
            chain = FINGER_CHAINS[finger]
            for joint_i in range(3):
                v1 = hw[chain[joint_i + 1]] - hw[chain[joint_i]]
                v2 = hw[chain[joint_i + 2]] - hw[chain[joint_i + 1]]
                FINGER_FLEX[side][fi, finger_i, joint_i] = angle_deg(v1, v2)
    FINGER_FLEX[side] = ema_smooth(FINGER_FLEX[side], ALPHA)

# CURLS: mean flexion per finger [rad], consumed by both the MuJoCo finger
# hinges and the VRMA finger curls
CURLS = {side: np.radians(FINGER_FLEX[side].mean(axis=2)) for side in ["Left", "Right"]}

BODY_ANGLE_DEFS = {
    "left_elbow_flexion": ("LEFT_SHOULDER", "LEFT_ELBOW", "LEFT_WRIST"),
    "right_elbow_flexion": ("RIGHT_SHOULDER", "RIGHT_ELBOW", "RIGHT_WRIST"),
    "left_knee_flexion": ("LEFT_HIP", "LEFT_KNEE", "LEFT_ANKLE"),
    "right_knee_flexion": ("RIGHT_HIP", "RIGHT_KNEE", "RIGHT_ANKLE"),
    "left_shoulder_abduction": ("LEFT_HIP", "LEFT_SHOULDER", "LEFT_ELBOW"),
    "right_shoulder_abduction": ("RIGHT_HIP", "RIGHT_SHOULDER", "RIGHT_ELBOW"),
}
BODY_ANGLES = {name: np.zeros(NUM_FRAMES) for name in BODY_ANGLE_DEFS}
for fi in range(NUM_FRAMES):
    for name, (a, b, c) in BODY_ANGLE_DEFS.items():
        BODY_ANGLES[name][fi] = angle_deg(POSE_C[fi][P[a]] - POSE_C[fi][P[b]],
                                          POSE_C[fi][P[c]] - POSE_C[fi][P[b]])

# =====================================================================
# 4-E. Interaction candidates + object 3D proxy
#      Everything produced here is a HEURISTIC CANDIDATE, not ground truth.
#      Column names and captions keep the word "candidate" for that reason.
# =====================================================================
def hand_image_center(fi, side):
    if HAND_VALID[side].any():
        return HAND_N_RAW[side][fi][:, :2].mean(axis=0)
    return POSE_IMG[fi][P[f"{side.upper()}_WRIST"]][:2]


FRAME_INTERACTIONS = [[] for _ in range(NUM_FRAMES)]   # per-frame view for frames.jsonl
FRAME_PHASE = [{"action": BEH["task"].replace("_", " ").title(),
                "phase": "Idle", "hand": "-"} for _ in range(NUM_FRAMES)]
INTERACTION_SEGMENTS = []
PRIMARY_HAND = None
OBJ_PROXY_C = None
OBJ_PROXY_SOURCE = ["none"] * NUM_FRAMES

if PRIMARY_TRACK is not None and BEH["enable_interaction_candidates"]:
    track = PRIMARY_TRACK
    hand_dist = {}
    for side in ["Left", "Right"]:
        dist = np.full(NUM_FRAMES, np.nan)
        for fi in range(NUM_FRAMES):
            if not track["interp_valid"][fi]:
                continue
            scale = max(track["sizes"][fi].max(), 0.05)
            dist[fi] = np.linalg.norm(hand_image_center(fi, side)
                                      - track["centers"][fi]) / scale
        hand_dist[side] = dist
    min_dist = {s: np.nanmin(hand_dist[s]) if np.isfinite(hand_dist[s]).any()
                else np.inf for s in ["Left", "Right"]}
    PRIMARY_HAND = min(min_dist, key=min_dist.get)
    if not math.isfinite(min_dist[PRIMARY_HAND]):
        PRIMARY_HAND = "Right"
    dist = hand_dist[PRIMARY_HAND]

    contact_th = BEH["contact_distance"] * 2.5  # threshold on object-size-normalised distance
    release_th = contact_th * 1.6
    window = BEH["reach_window"]
    # State-machine design notes:
    # - A NaN distance means the object is occluded. If contact happened just
    #   before, we keep carrying: during a real lift the hand covers the
    #   object, so losing the detection is the normal case, not an error.
    # - Release needs hysteresis -- the object must be visible again AND far
    #   away for two consecutive frames. Without it the phase chatters
    #   between grasp/carry/release on single-frame detector noise.
    OPEN_CURL_TH = 0.35        # mean finger curl [rad] below this = hand is open
    OPEN_RELEASE_FRAMES = 3    # this many open frames in a row confirms release
    carrying = False
    carrying_per_frame = np.zeros(NUM_FRAMES, dtype=bool)
    grasp_onset, away_streak, open_streak, release_at = -10, 0, 0, -10
    phase_per_frame = []
    for fi in range(NUM_FRAMES):
        d = dist[fi]
        visible = bool(np.isfinite(d))
        moving = bool(track["moving"][fi])
        past = dist[max(0, fi - window):fi]
        recent_contact = bool(np.isfinite(past).any()
                              and np.nanmin(past[np.isfinite(past)]) < contact_th)
        # Evidence that the hand is open -- available even while the object
        # itself is occluded:
        #   1) gesture is explicitly Open_Palm  -> open
        #   2) gesture is a closed form         -> closed (beats finger curl)
        #   3) gesture unknown                  -> fall back to finger curl
        gesture = frames[fi]["gestures"].get(PRIMARY_HAND, {}).get("gesture")
        if gesture == "Open_Palm":
            hand_open = True
        elif gesture in ("Closed_Fist", "Thumb_Up", "Thumb_Down",
                         "Pointing_Up", "Victory", "ILoveYou"):
            hand_open = False
        else:
            hand_open = bool(HAND_VALID[PRIMARY_HAND][fi]
                             and float(CURLS[PRIMARY_HAND][fi].mean())
                             < OPEN_CURL_TH)
        if not carrying:
            if visible and d < contact_th and moving:
                carrying, grasp_onset = True, fi
                away_streak = open_streak = 0
            elif not visible and recent_contact:
                # Object vanished right after contact = grasped and hidden
                carrying, grasp_onset = True, fi
                away_streak = open_streak = 0
        else:
            if visible and d > release_th:
                away_streak += 1
            elif visible:
                away_streak = 0
            # Even with the object invisible, a hand that stays open means
            # it was put down. Without this, carry would last forever when
            # the object is never re-detected after being placed.
            open_streak = open_streak + 1 if hand_open else 0
            if away_streak >= 2 or open_streak >= OPEN_RELEASE_FRAMES:
                carrying, release_at = False, fi
        carrying_per_frame[fi] = carrying

        cands = []
        if carrying:
            if fi - grasp_onset < 2:
                cands.append(("grasp_candidate", 0.8))
            else:
                cands.append(("carry_candidate", 0.85))
        elif fi - release_at < 2 and release_at >= 0:
            cands.append(("release_candidate", 0.7))
        elif visible and d < contact_th:
            cands.append(("contact_candidate", 0.75))
        elif visible and np.isfinite(past).any() \
                and np.nanmean(past) - d > 0.02 and d < contact_th * 3:
            cands.append(("reach_candidate", 0.6))
        phase = cands[0][0] if cands else "idle"
        phase_per_frame.append(phase)
        for cand, score in cands:
            FRAME_INTERACTIONS[fi].append({
                "type": cand, "hand": PRIMARY_HAND.lower(),
                "object_id": track["track_id"], "score": score})
        FRAME_PHASE[fi] = {
            "action": BEH["task"].replace("_", " ").title(),
            "phase": phase.replace("_candidate", "").title() if cands else "Idle",
            "hand": PRIMARY_HAND,
        }

    # Collapse per-frame phases into segments for interaction_events.csv
    seg_start, seg_type = 0, phase_per_frame[0]
    for fi in range(1, NUM_FRAMES + 1):
        current = phase_per_frame[fi] if fi < NUM_FRAMES else None
        if current != seg_type:
            if seg_type != "idle":
                INTERACTION_SEGMENTS.append({
                    "type": seg_type, "hand": PRIMARY_HAND.lower(),
                    "object_id": track["track_id"],
                    "start_frame": seg_start, "end_frame": fi - 1,
                    "start_sec": round(frames[seg_start]["timestamp_sec"], 3),
                    "end_sec": round(frames[fi - 1]["timestamp_sec"], 3)})
            seg_start, seg_type = fi, current

    # ---- Object 3D proxy ----
    # We never fabricate a 3D position silently: every frame carries a
    # position_source telling the consumer how that number was obtained.
    OBJ_PROXY_C = np.zeros((NUM_FRAMES, 3))
    fww = MJ["frame_world_width"]
    last_known = None   # hold the last position while the object stays unseen
    for fi in range(NUM_FRAMES):
        cx, cy = track["centers"][fi]
        base = np.array([(cx - 0.5) * fww, (0.5 - cy) * fww * 0.55 + 0.9, 0.30])
        if carrying_per_frame[fi]:
            wrist_img = POSE_IMG[fi][P[f"{PRIMARY_HAND.upper()}_WRIST"]]
            OBJ_PROXY_C[fi] = [(wrist_img[0] - 0.5) * fww,
                               (0.5 - wrist_img[1]) * fww * 0.55 + 0.9, 0.30]
            OBJ_PROXY_SOURCE[fi] = ("estimated_from_hand"
                                    if track["valid_raw"][fi]
                                    else "estimated_from_hand_occlusion")
            last_known = OBJ_PROXY_C[fi].copy()
        elif track["valid_raw"][fi]:
            OBJ_PROXY_C[fi] = base
            OBJ_PROXY_SOURCE[fi] = "detected_2d"
            last_known = base.copy()
        elif last_known is not None:
            # Invisible and not carried: hold last known instead of jumping
            OBJ_PROXY_C[fi] = last_known
            OBJ_PROXY_SOURCE[fi] = "last_known_position"
        else:
            OBJ_PROXY_C[fi] = base
            OBJ_PROXY_SOURCE[fi] = "fixed_depth_proxy"
    # Carried frames already follow the smoothed wrist, so do not smooth
    # them twice -- double smoothing makes the object lag the hand by 1-2 frames.
    smoothed = OBJ_PROXY_C.copy()
    for fi in range(1, NUM_FRAMES):
        if carrying_per_frame[fi]:
            smoothed[fi] = OBJ_PROXY_C[fi]
        else:
            smoothed[fi] = (ALPHA * OBJ_PROXY_C[fi]
                            + (1.0 - ALPHA) * smoothed[fi - 1])
    OBJ_PROXY_C = smoothed

print("Interaction segments:",
      [f"{s['type']}({s['start_sec']}-{s['end_sec']}s)" for s in INTERACTION_SEGMENTS]
      or "none")

# =====================================================================
# 4-F. Motion metrics + quality report
# =====================================================================
def path_length(points):
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())


wrist_speeds = {}
for side in ["Left", "Right"]:
    wrist = POSE_C[:, P[f"{side.upper()}_WRIST"]]
    speed = np.linalg.norm(np.diff(wrist, axis=0), axis=1) / DT
    wrist_speeds[side] = speed

jerk = np.diff(wrist_speeds.get(PRIMARY_HAND or "Right", wrist_speeds["Right"]))
MOTION_METRICS = {
    "duration_sec": round(frames[-1]["timestamp_sec"], 3),
    "left_hand_path_length_m": round(path_length(POSE_C[:, P["LEFT_WRIST"]]), 3),
    "right_hand_path_length_m": round(path_length(POSE_C[:, P["RIGHT_WRIST"]]), 3),
    "left_wrist_average_speed": round(float(wrist_speeds["Left"].mean()), 3),
    "right_wrist_average_speed": round(float(wrist_speeds["Right"].mean()), 3),
    "left_wrist_max_speed": round(float(wrist_speeds["Left"].max()), 3),
    "right_wrist_max_speed": round(float(wrist_speeds["Right"].max()), 3),
    "elbow_range_of_motion_deg": round(float(
        max(np.ptp(BODY_ANGLES["left_elbow_flexion"]),
            np.ptp(BODY_ANGLES["right_elbow_flexion"]))), 1),
    "motion_smoothness": round(float(1.0 / (1.0 + np.abs(jerk).mean())), 3),
    "left_right_coordination": (lambda c: round(float(c), 3)
                                if np.isfinite(c) else None)(
        np.corrcoef(wrist_speeds["Left"], wrist_speeds["Right"])[0, 1]
        if NUM_FRAMES > 2 else np.nan),
    "pose_detection_rate": round(COVERAGE["pose"] / NUM_FRAMES, 3),
    "hand_detection_rate": round(
        max(COVERAGE["left_hand"], COVERAGE["right_hand"]) / NUM_FRAMES, 3),
}
OBJECT_METRICS = {}
if PRIMARY_TRACK is not None:
    centers = PRIMARY_TRACK["centers"]
    OBJECT_METRICS = {
        "track_id": PRIMARY_TRACK["track_id"],
        "label": PRIMARY_TRACK["label"],
        "object_visible_duration_sec": round(len(PRIMARY_TRACK["obs"]) * DT, 2),
        "object_path_length_2d": round(path_length(centers), 3),
        "object_displacement_2d": round(float(np.linalg.norm(centers[-1] - centers[0])), 3),
        "object_average_speed_2d": round(float(PRIMARY_TRACK["speed"].mean()), 3),
        "object_max_speed_2d": round(float(PRIMARY_TRACK["speed"].max()), 3),
        "object_detection_rate": round(len(PRIMARY_TRACK["obs"]) / NUM_FRAMES, 3),
    }
INTERACTION_METRICS = {
    "minimum_hand_object_distance": (round(float(np.nanmin(dist)), 3)
                                     if PRIMARY_TRACK is not None else None),
    "grasp_candidate_count": sum(1 for s in INTERACTION_SEGMENTS
                                 if s["type"] == "grasp_candidate"),
    "carry_duration_sec": round(sum(s["end_sec"] - s["start_sec"]
                                    for s in INTERACTION_SEGMENTS
                                    if s["type"] == "carry_candidate"), 2),
    "release_candidate_count": sum(1 for s in INTERACTION_SEGMENTS
                                   if s["type"] == "release_candidate"),
}
QUALITY = {
    "overall_quality": round(float(np.mean([
        COVERAGE["pose"] / NUM_FRAMES,
        max(COVERAGE["left_hand"], COVERAGE["right_hand"]) / NUM_FRAMES,
        COVERAGE["face"] / NUM_FRAMES,
        (COVERAGE["objects"] / NUM_FRAMES) if TRACKS else 1.0])), 3),
    "pose_coverage": round(COVERAGE["pose"] / NUM_FRAMES, 3),
    "left_hand_coverage": round(COVERAGE["left_hand"] / NUM_FRAMES, 3),
    "right_hand_coverage": round(COVERAGE["right_hand"] / NUM_FRAMES, 3),
    "face_coverage": round(COVERAGE["face"] / NUM_FRAMES, 3),
    "object_coverage": round(COVERAGE["objects"] / NUM_FRAMES, 3),
    "missing_frame_ratio": round(1.0 - COVERAGE["pose"] / NUM_FRAMES, 3),
    "notes": [],
}
if QUALITY["pose_coverage"] < 0.8:
    QUALITY["notes"].append("Low pose coverage. Keep the full body inside the frame.")
if TRACKS and QUALITY["object_coverage"] < 0.5:
    QUALITY["notes"].append("Low object coverage. Use a clearly visible object (bottle/cup).")

print(json.dumps({"motion_metrics": MOTION_METRICS,
                  "interaction_metrics": INTERACTION_METRICS}, indent=2))


In [ ]:
# =====================================================================
# [5] Write the Behavior Dataset to disk
#     04_behavior_dataset/ holds human / objects / interactions / timeline /
#     metrics / quality plus a manifest and summary. This directory IS the
#     master data -- every other output in this notebook is derived from it.
# =====================================================================
WRITTEN = []


def note_written(path, rows=None):
    rel = str(Path(path).relative_to(DATASET_DIR))
    WRITTEN.append(rel + (f" ({rows} rows)" if rows is not None else ""))


# ---------------- human/pose_landmarks.csv ----------------
rows = []
for fi, record in enumerate(frames):
    detected = record["pose"] is not None
    for li, name in enumerate(POSE_LANDMARK_NAMES):
        n = record["pose"]["normalized"][li] if detected else [None] * 3
        w = (record["pose"]["world"][li]
             if detected and record["pose"]["world"] is not None else [None] * 3)
        v = record["pose"]["visibility"][li] if detected else [None, None]
        rows.append([fi, record["timestamp_ms"], li, name,
                     *(n if detected else [None]*3),
                     *(w if w is not None else [None]*3),
                     v[0] if detected else None, v[1] if detected else None])
pose_df = pd.DataFrame(rows, columns=[
    "frame", "timestamp_ms", "landmark_id", "landmark_name",
    "x", "y", "z", "world_x", "world_y", "world_z", "visibility", "presence"])
pose_df.to_csv(DATASET_DIR / "human/pose_landmarks.csv", index=False)
note_written(DATASET_DIR / "human/pose_landmarks.csv", len(pose_df))

# ---------------- human/hand_landmarks.csv ----------------
rows = []
for fi, record in enumerate(frames):
    for side in ["Left", "Right"]:
        if side not in record["hands"]:
            continue
        hand = record["hands"][side]
        for li, name in enumerate(HAND_LANDMARK_NAMES):
            rows.append([fi, record["timestamp_ms"], side.lower(), li, name,
                         *hand["normalized"][li], *hand["world"][li],
                         hand["score"]])
hand_df = pd.DataFrame(rows, columns=[
    "frame", "timestamp_ms", "hand", "landmark_id", "landmark_name",
    "x", "y", "z", "world_x", "world_y", "world_z", "handedness_score"])
hand_df.to_csv(DATASET_DIR / "human/hand_landmarks.csv", index=False)
note_written(DATASET_DIR / "human/hand_landmarks.csv", len(hand_df))

# ---------------- human/face_landmarks.jsonl + face_blendshapes.csv ----------------
with open(DATASET_DIR / "human/face_landmarks.jsonl", "w") as fp:
    for fi, record in enumerate(frames):
        if record["face"] is None:
            continue
        fp.write(json.dumps({
            "frame": fi, "timestamp_ms": record["timestamp_ms"],
            "landmarks": np.round(record["face"]["landmarks"], 5).tolist(),
            "transform": (np.round(record["face"]["transform"], 5).tolist()
                          if record["face"]["transform"] is not None else None),
        }) + "\n")
note_written(DATASET_DIR / "human/face_landmarks.jsonl", COVERAGE["face"])

blend_names = sorted({name for f in frames if f["face"]
                      for name in f["face"]["blendshapes"]})
rows = []
for fi, record in enumerate(frames):
    if record["face"] is None:
        continue
    rows.append([fi, record["timestamp_ms"]] +
                [round(record["face"]["blendshapes"].get(n, 0.0), 4)
                 for n in blend_names])
pd.DataFrame(rows, columns=["frame", "timestamp_ms"] + blend_names).to_csv(
    DATASET_DIR / "human/face_blendshapes.csv", index=False)
note_written(DATASET_DIR / "human/face_blendshapes.csv", len(rows))

# ---------------- human/gestures.csv ----------------
rows = [[fi, r["timestamp_ms"], side.lower(), g["gesture"], round(g["score"], 3)]
        for fi, r in enumerate(frames) for side, g in r["gestures"].items()]
pd.DataFrame(rows, columns=["frame", "timestamp_ms", "hand", "gesture",
                            "score"]).to_csv(
    DATASET_DIR / "human/gestures.csv", index=False)
note_written(DATASET_DIR / "human/gestures.csv", len(rows))

# ---------------- human/bone_rotations.csv (quaternions as [x, y, z, w]) ----------------
rows = []
for fi, record in enumerate(frames):
    for bone in BONE_ORDER:
        w, x, y, z = BONE_ROT[bone][fi]
        rows.append([fi, record["timestamp_ms"], bone,
                     round(x, 5), round(y, 5), round(z, 5), round(w, 5)])
bone_df = pd.DataFrame(rows, columns=["frame", "timestamp_ms", "bone",
                                      "rot_x", "rot_y", "rot_z", "rot_w"])
bone_df.to_csv(DATASET_DIR / "human/bone_rotations.csv", index=False)
note_written(DATASET_DIR / "human/bone_rotations.csv", len(bone_df))

# ---------------- human/joint_angles.csv ----------------
rows = []
for fi, record in enumerate(frames):
    entry = [fi, record["timestamp_ms"]]
    entry += [round(BODY_ANGLES[name][fi], 2) for name in BODY_ANGLE_DEFS]
    for side in ["Left", "Right"]:
        for finger_i, finger in enumerate(FINGER_ORDER):
            for joint_i in range(3):
                entry.append(round(float(FINGER_FLEX[side][fi, finger_i, joint_i]), 2))
    rows.append(entry)
angle_columns = (["frame", "timestamp_ms"] + list(BODY_ANGLE_DEFS) +
                 [f"{s.lower()}_{f}_J{j+1}_flexion_deg"
                  for s in ["Left", "Right"] for f in FINGER_ORDER for j in range(3)])
pd.DataFrame(rows, columns=angle_columns).to_csv(
    DATASET_DIR / "human/joint_angles.csv", index=False)
note_written(DATASET_DIR / "human/joint_angles.csv", len(rows))

# ---------------- objects/ ----------------
rows = [[fi, r["timestamp_ms"], o["label"], round(o["score"], 3),
         round(o["x_min"], 4), round(o["y_min"], 4),
         round(o["width"], 4), round(o["height"], 4)]
        for fi, r in enumerate(frames) for o in r["objects"]]
pd.DataFrame(rows, columns=["frame", "timestamp_ms", "label", "score",
                            "bbox_x", "bbox_y", "bbox_width",
                            "bbox_height"]).to_csv(
    DATASET_DIR / "objects/object_detections.csv", index=False)
note_written(DATASET_DIR / "objects/object_detections.csv", len(rows))

rows = []
for track in TRACKS:
    for fi in range(NUM_FRAMES):
        if not track["interp_valid"][fi]:
            continue
        det = track["obs"].get(fi)
        rows.append([fi, frames[fi]["timestamp_ms"], track["track_id"],
                     track["label"],
                     (det or {}).get("detector_label", track["label"]),
                     track.get("role", "target"),
                     round(float(track["centers"][fi][0]), 4),
                     round(float(track["centers"][fi][1]), 4),
                     round(float(track["velocity"][fi][0]), 4),
                     round(float(track["velocity"][fi][1]), 4),
                     round(float(track["speed"][fi]), 4),
                     "moving" if track["moving"][fi] else "stationary",
                     bool(track["valid_raw"][fi])])
pd.DataFrame(rows, columns=["frame", "timestamp_ms", "track_id", "label",
                            "detector_label", "role",
                            "center_x", "center_y", "velocity_x", "velocity_y",
                            "speed", "motion_state", "detected"]).to_csv(
    DATASET_DIR / "objects/object_tracks.csv", index=False)
note_written(DATASET_DIR / "objects/object_tracks.csv", len(rows))

# ---------------- interactions/interaction_events.csv (candidates only) ----------------
pd.DataFrame(INTERACTION_SEGMENTS or [], columns=[
    "type", "hand", "object_id", "start_frame", "end_frame",
    "start_sec", "end_sec"]).to_csv(
    DATASET_DIR / "interactions/interaction_events.csv", index=False)
note_written(DATASET_DIR / "interactions/interaction_events.csv",
             len(INTERACTION_SEGMENTS))

# ---------------- timeline/frames.jsonl ----------------
# One line = one complete frame record, and the single most important file
# in this dataset. A row-oriented view where vision, language, motion, phase,
# objects and interaction candidates all live on the same row -- which is
# exactly what makes it loadable as ML training data (see Demo B).
# The CSVs above are the column-oriented view of the same master data; use
# them for analysis and adapters that only need one series.
# Only the 478-point face mesh is referenced rather than inlined, for size.


def _round_list(array, digits=4):
    return np.round(np.asarray(array, dtype=float), digits).tolist()


def build_frame_record(fi, record):
    human = {
        "pose_canonical": (_round_list(POSE_C[fi]) if POSE_VALID[fi] else None),
        "hips_position": _round_list(HIPS_POS_C[fi]),
        "bone_rotations_xyzw": {
            bone: _round_list(np.roll(BONE_ROT[bone][fi], -1), 5)
            for bone in BONE_ROT},
        "joint_angles_deg": {name: round(float(BODY_ANGLES[name][fi]), 2)
                             for name in BODY_ANGLES},
        "finger_curls_rad": {
            side.lower(): (_round_list(CURLS[side][fi], 3)
                           if HAND_VALID[side][fi] else None)
            for side in ["Left", "Right"]},
        "hands_canonical": {
            side.lower(): (_round_list(HAND_C[side][fi])
                           if HAND_VALID[side][fi] else None)
            for side in ["Left", "Right"]},
        "face_blendshapes": ({k: round(v, 3)
                              for k, v in record["face"]["blendshapes"].items()}
                             if record["face"] else None),
        "face_landmarks_ref": ("human/face_landmarks.jsonl"
                               if record["face"] else None),
        "gestures": {s.lower(): {"gesture": g["gesture"],
                                 "score": round(g["score"], 3)}
                     for s, g in record["gestures"].items()},
    }
    objects = []
    for track in TRACKS:
        if not track["interp_valid"][fi]:
            continue
        entry = {
            "track_id": track["track_id"],
            "label": track["label"],
            "detector_label": (track["obs"].get(fi) or {}).get(
                "detector_label", track["label"]),
            "role": track.get("role", "target"),
            "center_normalized": _round_list(track["centers"][fi]),
            "state": "moving" if track["moving"][fi] else "stationary",
        }
        if track is PRIMARY_TRACK and OBJ_PROXY_C is not None:
            entry["proxy_canonical"] = _round_list(OBJ_PROXY_C[fi])
            entry["position_source"] = OBJ_PROXY_SOURCE[fi]
        objects.append(entry)
    return {
        "frame": fi,
        "source_frame_index": record["source_frame_index"],
        "frame_image": FRAME_IMAGE_PATHS[fi],
        "timestamp_ms": record["timestamp_ms"],
        "timestamp_sec": round(record["timestamp_sec"], 4),
        "human": human,
        "objects": objects,
        "interactions": FRAME_INTERACTIONS[fi],
        "phase": FRAME_PHASE[fi],
        "caption": None,  # filled in by [5.5] when captions are generated
    }


# ---- Export one image per analysed frame ----
# These are the vision half of each training tuple. Straight source frames,
# just downscaled -- nothing is rendered or synthesised.
FRAME_IMAGE_PATHS = [None] * NUM_FRAMES
if CONFIG["video"].get("export_frame_images", False):
    frames_img_dir = DATASET_DIR / "timeline/frames"
    frames_img_dir.mkdir(exist_ok=True)
    _cap = cv2.VideoCapture(str(SOURCE_VIDEO))
    _w = CONFIG["video"].get("frame_image_width", 320)
    for fi, record in enumerate(frames):
        _cap.set(cv2.CAP_PROP_POS_FRAMES, record["source_frame_index"])
        ok, img = _cap.read()
        if not ok:
            continue
        h = int(img.shape[0] * _w / img.shape[1])
        img = cv2.resize(img, (_w, h))
        rel = f"timeline/frames/{fi:06d}.jpg"
        cv2.imwrite(str(DATASET_DIR / rel), img,
                    [cv2.IMWRITE_JPEG_QUALITY, 82])
        FRAME_IMAGE_PATHS[fi] = rel
    _cap.release()
    _count = sum(1 for p in FRAME_IMAGE_PATHS if p)
    _size = sum((DATASET_DIR / p).stat().st_size
                for p in FRAME_IMAGE_PATHS if p) / 1024 / 1024
    note_written(frames_img_dir, _count)
    print(f"  frame images: {_count} ({_size:.1f} MB)")

with open(DATASET_DIR / "timeline/frames.jsonl", "w") as fp:
    for fi, record in enumerate(frames):
        fp.write(json.dumps(build_frame_record(fi, record)) + "\n")
note_written(DATASET_DIR / "timeline/frames.jsonl", NUM_FRAMES)

# ---------------- metrics / quality ----------------
pd.DataFrame([MOTION_METRICS]).to_csv(
    DATASET_DIR / "metrics/motion_metrics.csv", index=False)
pd.DataFrame([{**OBJECT_METRICS, **INTERACTION_METRICS}]).to_csv(
    DATASET_DIR / "metrics/object_metrics.csv", index=False)
(DATASET_DIR / "quality/quality.json").write_text(
    json.dumps(QUALITY, indent=2), encoding="utf-8")
note_written(DATASET_DIR / "metrics/motion_metrics.csv")
note_written(DATASET_DIR / "metrics/object_metrics.csv")
note_written(DATASET_DIR / "quality/quality.json")

# ---------------- manifest.json ----------------
MANIFEST = {
    "dataset_version": "0.2.0",
    "demo_name": "human_behavior_demo_2_0",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_video": "source_video.mp4",
    "source_fps": round(SOURCE_FPS, 3),
    "fps": round(EFFECTIVE_FPS, 3),
    "frame_count": NUM_FRAMES,
    "duration_sec": round(frames[-1]["timestamp_sec"], 2),
    "person_count": 1,
    "face_landmark_count": FACE_LANDMARK_COUNT,
    "coordinate_system": {
        "raw": "mediapipe (x=image right, y=image down, z=away from camera)",
        "canonical": "body_capture (x=right, y=up, z=toward camera / person faces +z)",
        "adapters": {"mujoco": "z-up, person faces -y", "unity_vrm": "gltf y-up"},
    },
    "mediapipe": {task: bool(TASK_DELEGATE_LOG.get(task, {}).get("actual_delegate")
                             not in (None, "unavailable"))
                  for task in ["pose", "hands", "face", "gesture", "object"]},
    "outputs": {"mediapipe_overlay": True, "mujoco": True,
                "vrma": CONFIG["vrm"]["export_vrma"], "behavior_dataset": True},
}
(DATASET_DIR / "manifest.json").write_text(
    json.dumps(MANIFEST, indent=2), encoding="utf-8")
note_written(DATASET_DIR / "manifest.json")

# ---------------- behavior_summary.json ----------------
BEHAVIOR_SUMMARY = {
    "behavior_id": "behavior_0001",
    "task": BEH["task"],
    "description": (f"Person interacts with a {PRIMARY_TRACK['label']} "
                    "(pick / move / place candidates)." if PRIMARY_TRACK
                    else "Person motion without detected object."),
    "primary_hand": (PRIMARY_HAND or "right").lower(),
    "primary_object": ({"track_id": PRIMARY_TRACK["track_id"],
                        "label": PRIMARY_TRACK["label"]}
                       if PRIMARY_TRACK else None),
    "start_sec": 0.0,
    "end_sec": round(frames[-1]["timestamp_sec"], 2),
    "events": sorted({s["type"] for s in INTERACTION_SEGMENTS}),
}
(DATASET_DIR / "behavior_summary.json").write_text(
    json.dumps(BEHAVIOR_SUMMARY, indent=2), encoding="utf-8")
note_written(DATASET_DIR / "behavior_summary.json")

# ---------------- dataset README.md ----------------
(DATASET_DIR / "README.md").write_text(f"""# Human Behavior Dataset

Generated by Human Behavior Demo 2.0 from `source_video.mp4`.

- Start with `behavior_summary.json` (what happened) and `manifest.json` (index).
- `human/` : pose / hand / face landmarks, gestures, bone rotations, joint angles
- `objects/` : raw detections and tracked object motion (2D image space)
- `interactions/` : heuristic interaction *candidates* (not ground truth)
- `timeline/frames.jsonl` : one line per frame, machine-readable unified timeline
- `metrics/`, `quality/` : objective motion metrics and dataset quality
- `adapters/` : engine-specific derived data (MuJoCo qpos, VRM bone rotations)

Raw MediaPipe values are preserved. Canonical bone rotations are quaternions
`[x, y, z, w]` relative to a T-pose rest, engine independent.
Derived object 3D positions always carry `position_source`
(`detected_2d` / `estimated_from_hand` / `fixed_depth_proxy`).
""", encoding="utf-8")
note_written(DATASET_DIR / "README.md")

# Top-level README shipped with the outputs
(PROJECT_DIR / "README.md").write_text(
    "# Human Behavior Demo 2.0\n\n"
    "One source video becomes one reusable Human Behavior Dataset for\n"
    "avatars, simulation, analytics, and Physical AI.\n\n"
    "- output/01_mediapipe_overlay ... MediaPipe overlay video\n"
    "- output/02_mujoco ............. humanoid.xml + motion.npz + replay\n"
    "- output/03_unity_vrm .......... motion.vrma (play locally with Unity/UniVRM)\n"
    "- output/04_behavior_dataset ... master Common Behavior Data\n"
    "- output/05_comparison ......... three-screen comparison video\n",
    encoding="utf-8")

print("Behavior Dataset written:")
for line in WRITTEN:
    print("  -", line)


In [ ]:
# =====================================================================
# [5.5] Temporal captions via the Gemini API (optional cell)
#     This is what supplies the *language* half of the behavior data.
#     Keyframes plus behavior context go to Gemini; the captions come back
#     into 04_behavior_dataset/captions/ and are injected per-frame into
#     timeline/frames.jsonl.
#     The key is read from Colab Secrets (GEMINI_API_KEY) and never stored.
#     Without a key this cell simply skips -- every other output still works.
# =====================================================================
import base64
import urllib.request

# Tried in order; cheap flash-lite models first.
CAPTION_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite",
                  "gemini-2.5-flash-lite", "flash-lite-latest"]
CAPTION_MIN_WINDOW_SECONDS = 0.6   # shorter phase runs are absorbed into the previous window
CAPTION_MAX_WINDOW_SECONDS = 4.0   # longer windows (e.g. a long Idle) are split evenly
CAPTION_MAX_WINDOWS = 8            # cap on windows; the shortest get merged away
CAPTION_IMAGES_PER_WINDOW = 3      # first / middle / last frame of each window
CAPTION_IMAGE_WIDTH = 480          # downscaled before upload, to keep cost low

# ------------------------- API key (Colab Secrets) -------------------------
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

TEMPORAL_CAPTIONS = None
if not GEMINI_API_KEY:
    print("GEMINI_API_KEY not found -- skipping caption generation.")
    print("To enable: open the key icon in the Colab sidebar, add a secret")
    print("named GEMINI_API_KEY, turn on notebook access, and re-run this cell.")
else:
    # ------------------- Build time windows at phase boundaries -------------------
    # Windows follow the behavior phase runs from [4] rather than a fixed
    # clock. One caption then describes one action, which is what makes the
    # caption line up with the motion it is supposed to describe.
    runs = []
    for fi in range(NUM_FRAMES):
        phase = FRAME_PHASE[fi]["phase"]
        if runs and runs[-1]["phase"] == phase:
            runs[-1]["end"] = fi
        else:
            runs.append({"start": fi, "end": fi, "phase": phase})

    def run_seconds(run):
        return (frames[run["end"]]["timestamp_sec"]
                - frames[run["start"]]["timestamp_sec"])

    # Absorb runs that are too short to be a real action (detector flicker)
    absorbed = []
    for run in runs:
        if absorbed and run_seconds(run) < CAPTION_MIN_WINDOW_SECONDS:
            absorbed[-1]["end"] = run["end"]
        else:
            absorbed.append(run)
    # Absorption can leave two identical phases adjacent -- re-merge them
    merged_runs = []
    for run in absorbed:
        if merged_runs and merged_runs[-1]["phase"] == run["phase"]:
            merged_runs[-1]["end"] = run["end"]
        else:
            merged_runs.append(run)
    # Split runs that are too long (a long Idle) into equal parts
    windows = []
    for run in merged_runs:
        parts = max(1, int(math.ceil(run_seconds(run)
                                     / CAPTION_MAX_WINDOW_SECONDS)))
        count = run["end"] - run["start"] + 1
        for pi in range(parts):
            windows.append({
                "start": run["start"] + count * pi // parts,
                "end": run["start"] + count * (pi + 1) // parts - 1,
                "phase": run["phase"]})
    # Over the window cap: repeatedly fold the shortest window into a neighbour
    while len(windows) > CAPTION_MAX_WINDOWS:
        lengths = [w["end"] - w["start"] for w in windows]
        i = lengths.index(min(lengths))
        j = i - 1 if i > 0 else i + 1
        a, b = min(i, j), max(i, j)
        if lengths[b] > lengths[a]:
            windows[a]["phase"] = windows[b]["phase"]
        windows[a]["end"] = windows[b]["end"]
        del windows[b]

    def dominant(values):
        values = [v for v in values if v]
        if not values:
            return None
        return max(set(values), key=values.count)

    # Take first / middle / last frame of each window, so the model can see
    # the *direction* of the motion rather than a single ambiguous pose
    video = cv2.VideoCapture(str(SOURCE_VIDEO))
    window_infos = []
    for wi, win in enumerate(windows):
        indices = list(range(win["start"], win["end"] + 1))
        picks = sorted({indices[0], indices[len(indices) // 2], indices[-1]})
        picks = picks[:CAPTION_IMAGES_PER_WINDOW]
        jpegs = []
        for fi in picks:
            video.set(cv2.CAP_PROP_POS_FRAMES,
                      frames[fi]["source_frame_index"])
            success, image = video.read()
            if not success:
                continue
            height = int(image.shape[0] * CAPTION_IMAGE_WIDTH / image.shape[1])
            image = cv2.resize(image, (CAPTION_IMAGE_WIDTH, height))
            ok, buffer = cv2.imencode(".jpg", image,
                                      [cv2.IMWRITE_JPEG_QUALITY, 80])
            if ok:
                jpegs.append(base64.b64encode(buffer.tobytes()).decode("ascii"))
        if not jpegs:
            continue
        gestures = dominant([frames[i]["gestures"][s]["gesture"]
                             for i in indices for s in frames[i]["gestures"]])
        objects = sorted({t["label"] for t in TRACKS
                          if any(t["valid_raw"][i] for i in indices)})
        window_infos.append({
            "window_index": wi,
            "start_sec": round(frames[win["start"]]["timestamp_sec"], 3),
            "end_sec": round(frames[win["end"]]["timestamp_sec"], 3),
            "phase": win["phase"],
            "action": FRAME_PHASE[win["start"]]["action"],
            "hand": FRAME_PHASE[win["start"]]["hand"],
            "gesture": gestures, "objects": objects,
            "jpegs_b64": jpegs,
        })
    video.release()

    # ------------------------- Build the prompt -------------------------
    parts = [{"text": (
        "You are annotating a human behavior capture video for a robotics "
        "behavior dataset. The video is segmented into windows at behavior "
        "phase boundaries. For each window you receive up to 3 keyframes in "
        "temporal order (start -> middle -> end), plus heuristic context "
        "(phase / gesture / detected objects; these are candidates, not "
        "ground truth). Compare the keyframes to infer the direction of "
        "motion. For EACH window, write one factual English caption "
        "(max 20 words) describing what the person is doing, plus a natural "
        "Japanese translation. Describe only what is visible; do not invent "
        "objects or actions. Respond ONLY with a JSON array, one item per "
        "window, in this exact schema: "
        '[{"window_index": 0, "start_sec": 0.0, "end_sec": 2.0, '
        '"caption_en": "...", "caption_ja": "..."}]')}]
    for info in window_infos:
        parts.append({"text": (
            f"Window {info['window_index']} "
            f"({info['start_sec']}s - {info['end_sec']}s, "
            f"{len(info['jpegs_b64'])} keyframes in temporal order): "
            f"phase={info['phase']}, gesture={info['gesture']}, "
            f"objects={info['objects'] or 'none'}")})
        for jpeg in info["jpegs_b64"]:
            parts.append({"inline_data": {"mime_type": "image/jpeg",
                                          "data": jpeg}})

    request_body = json.dumps({
        "contents": [{"parts": parts}],
        "generationConfig": {"temperature": 0.4,
                             "response_mime_type": "application/json"},
    }).encode("utf-8")

    # ------------------- Call the API, trying each model in turn -------------------
    def parse_caption_json(text):
        text = text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text)

    response_text, used_model, last_error = None, None, None
    for model_name in CAPTION_MODELS:
        url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
               f"{model_name}:generateContent")
        request = urllib.request.Request(
            url, data=request_body, method="POST",
            headers={"Content-Type": "application/json",
                     "x-goog-api-key": GEMINI_API_KEY})
        try:
            with urllib.request.urlopen(request, timeout=180) as res:
                payload = json.loads(res.read().decode("utf-8"))
            response_text = "".join(
                part.get("text", "") for part in
                payload["candidates"][0]["content"]["parts"])
            used_model = model_name
            break
        except Exception as error:
            last_error = f"{model_name}: {error}"
            print(f"  {model_name} failed -> trying next model ({error})")

    # ------------------------- Save the result -------------------------
    if response_text is None:
        print("Caption generation failed (no other output is affected).")
        print("Last error:", last_error)
    else:
        try:
            caption_items = parse_caption_json(response_text)
        except Exception:
            caption_items = None
            print("Could not parse the response as JSON; storing raw text.")

        captions_dir = DATASET_DIR / "captions"
        captions_dir.mkdir(exist_ok=True)
        lookup = {info["window_index"]: info for info in window_infos}
        cleaned = []
        if caption_items:
            for item in caption_items:
                info = lookup.get(item.get("window_index"))
                if info is None:
                    continue
                cleaned.append({
                    "window_index": info["window_index"],
                    "start_sec": info["start_sec"],
                    "end_sec": info["end_sec"],
                    "phase_candidate": info["phase"],
                    "objects": info["objects"],
                    "caption_en": str(item.get("caption_en", "")).strip(),
                    "caption_ja": str(item.get("caption_ja", "")).strip(),
                })
        TEMPORAL_CAPTIONS = {
            "source": "gemini_api",
            "model": used_model,
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "window_count": len(window_infos),
            "window_basis": "behavior_phase_boundaries",
            "note": ("Captions are AI-generated descriptions, not ground "
                     "truth annotations."),
            "captions": cleaned,
        }
        if not cleaned:
            TEMPORAL_CAPTIONS["raw_response"] = response_text
        (captions_dir / "temporal_captions.json").write_text(
            json.dumps(TEMPORAL_CAPTIONS, indent=2, ensure_ascii=False),
            encoding="utf-8")
        if cleaned:
            pd.DataFrame(cleaned).to_csv(
                captions_dir / "temporal_captions.csv", index=False)

        # Register the captions in the manifest written by [5]
        manifest_path = DATASET_DIR / "manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["temporal_captions"] = {
            "model": used_model, "count": len(cleaned),
            "path": "captions/temporal_captions.json"}
        manifest_path.write_text(json.dumps(manifest, indent=2),
                                 encoding="utf-8")

        # Inject captions back into every row of timeline/frames.jsonl, so a
        # single line is a complete (vision + language + motion + phase) sample
        frames_jsonl = DATASET_DIR / "timeline/frames.jsonl"
        if cleaned and frames_jsonl.exists():
            records = [json.loads(line)
                       for line in frames_jsonl.read_text().splitlines()]
            for item in cleaned:
                for rec in records:
                    if item["start_sec"] <= rec["timestamp_sec"] < item["end_sec"] \
                            or rec is records[-1] and \
                            abs(rec["timestamp_sec"] - item["end_sec"]) < 1e-6:
                        rec["caption"] = {
                            "window_index": item["window_index"],
                            "en": item["caption_en"],
                            "ja": item["caption_ja"],
                            "source": "gemini_api",
                        }
            with open(frames_jsonl, "w") as fp:
                for rec in records:
                    fp.write(json.dumps(rec) + "\n")
            print("Captions injected into timeline/frames.jsonl")

        print(f"Wrote captions/temporal_captions.json "
              f"(model={used_model}, {len(cleaned)} windows)")
        for item in cleaned:
            print(f"  [{item['start_sec']:5.1f}s - {item['end_sec']:5.1f}s] "
                  f"{item['caption_en']}")


In [ ]:
# =====================================================================
# [6] Render the MediaPipe overlay video
#     Adapter #1: the behavior data drawn back onto the original pixels.
#     Pose / hands / face / objects / gestures / interaction candidates.
# =====================================================================
from collections import deque

OUTPUT_WIDTH = CONFIG["video"]["output_width"]
overlay_width = even(min(OUTPUT_WIDTH, SOURCE_WIDTH))
overlay_height = even(SOURCE_HEIGHT * overlay_width / SOURCE_WIDTH)

HAND_COLOR = {"Left": (255, 160, 60), "Right": (60, 160, 255)}   # BGR
POSE_COLOR = (90, 220, 90)
FACE_COLOR = (200, 200, 120)
OBJ_COLOR = (60, 60, 230)
TRAIL_COLOR = (80, 200, 255)

raw_overlay = WORK_DIR / "_overlay_raw.mp4"
writer = make_video_writer(raw_overlay, EFFECTIVE_FPS,
                           (overlay_width, overlay_height))
video = cv2.VideoCapture(str(SOURCE_VIDEO))
trail = deque(maxlen=int(EFFECTIVE_FPS * 2))
frame_lookup = {r["source_frame_index"]: r for r in frames}
source_frame_index = 0


def put_label(image, text, x, y, color=(255, 255, 255), scale=0.45):
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, 1, cv2.LINE_AA)


while True:
    success, frame_bgr = video.read()
    if not success or source_frame_index > MAX_SOURCE_FRAME:
        break
    record = frame_lookup.get(source_frame_index)
    source_frame_index += 1
    if record is None:
        continue
    fi = record["frame_index"]
    image = cv2.resize(frame_bgr, (overlay_width, overlay_height))
    W, H = overlay_width, overlay_height

    # ---- Face mesh (subsampled points, the full mesh is too dense to read) ----
    if record["face"] is not None:
        for point in record["face"]["landmarks"][::6]:
            cv2.circle(image, (int(point[0] * W), int(point[1] * H)),
                       1, FACE_COLOR, -1)

    # ---- Pose skeleton ----
    if record["pose"] is not None:
        pn = record["pose"]["normalized"]
        for a, b in POSE_BONES:
            cv2.line(image, (int(pn[a][0]*W), int(pn[a][1]*H)),
                     (int(pn[b][0]*W), int(pn[b][1]*H)), POSE_COLOR, 2)
        for point in pn:
            cv2.circle(image, (int(point[0]*W), int(point[1]*H)), 3,
                       POSE_COLOR, -1)

    # ---- Hands ----
    for side, hand in record["hands"].items():
        hn = hand["normalized"]
        color = HAND_COLOR[side]
        for a, b in HAND_BONES:
            cv2.line(image, (int(hn[a][0]*W), int(hn[a][1]*H)),
                     (int(hn[b][0]*W), int(hn[b][1]*H)), color, 2)
        for point in hn:
            cv2.circle(image, (int(point[0]*W), int(point[1]*H)), 2, color, -1)
        wx, wy = int(hn[0][0]*W), int(hn[0][1]*H)
        gesture = record["gestures"].get(side)
        text = f"{side} Hand"
        if gesture and gesture["gesture"] not in ("None", ""):
            text += f" | Gesture: {gesture['gesture']}"
        put_label(image, text, min(wx, W - 220), max(16, wy - 8), color)

    # ---- Objects (bbox + track id + trail + motion state) ----
    for track in TRACKS:
        if not track["valid_raw"][fi]:
            continue
        det = track["obs"].get(fi)
        if det is None:
            continue
        x0, y0 = int(det["x_min"]*W), int(det["y_min"]*H)
        x1 = int((det["x_min"] + det["width"]) * W)
        y1 = int((det["y_min"] + det["height"]) * H)
        cv2.rectangle(image, (x0, y0), (x1, y1), OBJ_COLOR, 2)
        state = "Moving" if track["moving"][fi] else "Stationary"
        put_label(image, f"Object: {track['label']}  Track: {track['track_id']}",
                  x0, max(14, y0 - 22), OBJ_COLOR)
        put_label(image, f"Conf: {det['score']:.2f}  State: {state}",
                  x0, max(28, y0 - 6), OBJ_COLOR)
        center = (int(track["centers"][fi][0]*W), int(track["centers"][fi][1]*H))
        cv2.circle(image, center, 4, OBJ_COLOR, -1)
        if track is PRIMARY_TRACK:
            trail.append(center)
    for t0, t1 in zip(list(trail), list(trail)[1:]):
        cv2.line(image, t0, t1, TRAIL_COLOR, 2)

    # ---- Behavior caption, top-left ----
    lines = ["Person 1"]
    interactions = FRAME_INTERACTIONS[fi]
    if interactions:
        top = interactions[0]
        lines.append(f"Interaction: "
                     f"{top['type'].replace('_', ' ').title()}")
    lines.append(f"Frame: {fi}   Time: {record['timestamp_sec']:05.2f} s")
    band_height = 18 * len(lines) + 10
    cv2.rectangle(image, (0, 0), (250, band_height), (0, 0, 0), -1)
    for li, line in enumerate(lines):
        put_label(image, line, 8, 18 + 18 * li)

    writer.write(image)

video.release()
writer.release()

OVERLAY_MP4 = OVERLAY_DIR / "mediapipe_overlay.mp4"
encode_h264(raw_overlay, OVERLAY_MP4, EFFECTIVE_FPS)
print("Wrote:", OVERLAY_MP4)
show_video(OVERLAY_MP4)


In [ ]:
# =====================================================================
# [7] MuJoCo adapter: export + render
#     Adapter #2. Writes humanoid.xml / motion.npz / object_trajectory.csv /
#     replay_mujoco.py, then renders mujoco_simulation.mp4 in a subprocess.
#     Note that the model and the motion are deliberately separate files:
#     motion.npz is a projection of the behavior data, not a MuJoCo original.
# =====================================================================

# --------------------- 7-A. Humanoid MJCF ---------------------
# A T-pose skeleton (arms along +/-X, legs along -Z, person facing -Y).
# Every body is defined with zero rotation, so each ball joint's qpos is
# exactly the canonical bone rotation after a coordinate transform -- no
# retargeting, no IK, no hidden fitting step.
FINGER_Y_LEFT = {"thumb": -0.042, "index": -0.024, "middle": -0.008,
                 "ring": 0.008, "little": 0.024}

MJ_BALL_ORDER = []      # ball joints, in qpos order
MJ_HINGE_ORDER = []     # hinge joints (fingers), in qpos order


def finger_bodies_xml(side):
    prefix = side.lower()
    sign = 1.0 if side == "Left" else -1.0
    axis = "0 1 0" if side == "Left" else "0 -1 0"
    parts = []
    for finger in FINGER_ORDER:
        y = FINGER_Y_LEFT[finger]
        joint_name = f"{prefix}_finger_{finger}"
        MJ_HINGE_ORDER.append(joint_name)
        parts.append(f"""
            <body name="{prefix}_{finger}" pos="{sign*0.088:.3f} {y:.3f} 0">
              <joint name="{joint_name}" type="hinge" axis="{axis}"
                     range="0 2.2" limited="true"/>
              <geom type="capsule" fromto="0 0 0 {sign*0.055:.3f} 0 0"
                    size="0.008" rgba="{FINGER_RGBA}"/>
            </body>""")
    return "".join(parts)


BODY_RGBA = "0.55 0.62 0.72 1"
ACCENT_RGBA = "0.85 0.45 0.25 1"
FINGER_RGBA = "0.80 0.80 0.85 1"


def ball(name):
    MJ_BALL_ORDER.append(name)
    return f'<joint name="{name}" type="ball"/>'


# Object proxy geometry, chosen from the primary object's class.
# Unity uses the same classification, so both engines show the same shape.
CYLINDER_LABELS = {"cup", "bottle", "wine glass", "vase", "bowl"}
SPHERE_LABELS = {"sports ball", "ball", "apple", "orange", "frisbee"}
_obj_label = PRIMARY_TRACK["label"] if PRIMARY_TRACK else None
if _obj_label in CYLINDER_LABELS:
    OBJECT_GEOM_XML = ('<geom type="cylinder" size="0.035 0.07" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')
elif _obj_label in SPHERE_LABELS:
    OBJECT_GEOM_XML = ('<geom type="sphere" size="0.05" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')
else:
    OBJECT_GEOM_XML = ('<geom type="box" size="0.05 0.05 0.09" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')

HUMANOID_XML = f"""<mujoco model="human_behavior_demo_2_0">
  <option gravity="0 0 0"/>
  <visual>
    <global offwidth="{CONFIG['mujoco']['render_width']}" offheight="{CONFIG['mujoco']['render_height']}"/>
    <quality shadowsize="1024" offsamples="{4 if HAS_GPU else 0}"/>
    <headlight ambient="0.45 0.45 0.45" diffuse="0.7 0.7 0.7"/>
  </visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" width="256" height="256"
             rgb1="0.20 0.24 0.30" rgb2="0.26 0.30 0.36"/>
    <material name="grid" texture="grid" texrepeat="6 6" reflectance="0.05"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="4 4 0.05" material="grid"/>
    <light pos="0 -2 3" dir="0 0.5 -1" diffuse="0.8 0.8 0.8"/>
    <body name="pelvis" pos="0 0 0.93">
      <freejoint name="root"/>
      <geom type="box" size="0.11 0.08 0.07" rgba="{BODY_RGBA}"/>
      <body name="spine" pos="0 0 0.10">
        {ball('spine')}
        <geom type="capsule" fromto="0 0 0 0 0 0.10" size="0.07" rgba="{BODY_RGBA}"/>
        <body name="chest" pos="0 0 0.12">
          {ball('chest')}
          <geom type="capsule" fromto="0 0 0 0 0 0.14" size="0.09" rgba="{BODY_RGBA}"/>
          <body name="neck" pos="0 0 0.20">
            {ball('neck')}
            <geom type="capsule" fromto="0 0 0 0 0 0.05" size="0.035" rgba="{BODY_RGBA}"/>
            <body name="head" pos="0 0 0.06">
              {ball('head')}
              <geom type="sphere" pos="0 0 0.08" size="0.09" rgba="{BODY_RGBA}"/>
              <geom type="sphere" pos="0.035 -0.075 0.09" size="0.012" rgba="0.1 0.1 0.1 1"/>
              <geom type="sphere" pos="-0.035 -0.075 0.09" size="0.012" rgba="0.1 0.1 0.1 1"/>
            </body>
          </body>
          <body name="left_shoulder" pos="0.08 0 0.14">
            <geom type="sphere" size="0.045" rgba="{BODY_RGBA}"/>
            <body name="left_upper_arm" pos="0.09 0 0">
              {ball('left_upper_arm')}
              <geom type="capsule" fromto="0 0 0 0.26 0 0" size="0.038" rgba="{BODY_RGBA}"/>
              <body name="left_lower_arm" pos="0.26 0 0">
                {ball('left_lower_arm')}
                <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.032" rgba="{BODY_RGBA}"/>
                <body name="left_hand" pos="0.25 0 0">
                  {ball('left_hand')}
                  <geom type="box" pos="0.045 0 0" size="0.045 0.032 0.012" rgba="{ACCENT_RGBA}"/>
                  {finger_bodies_xml('Left')}
                </body>
              </body>
            </body>
          </body>
          <body name="right_shoulder" pos="-0.08 0 0.14">
            <geom type="sphere" size="0.045" rgba="{BODY_RGBA}"/>
            <body name="right_upper_arm" pos="-0.09 0 0">
              {ball('right_upper_arm')}
              <geom type="capsule" fromto="0 0 0 -0.26 0 0" size="0.038" rgba="{BODY_RGBA}"/>
              <body name="right_lower_arm" pos="-0.26 0 0">
                {ball('right_lower_arm')}
                <geom type="capsule" fromto="0 0 0 -0.25 0 0" size="0.032" rgba="{BODY_RGBA}"/>
                <body name="right_hand" pos="-0.25 0 0">
                  {ball('right_hand')}
                  <geom type="box" pos="-0.045 0 0" size="0.045 0.032 0.012" rgba="{ACCENT_RGBA}"/>
                  {finger_bodies_xml('Right')}
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
      <body name="left_upper_leg" pos="0.09 0 -0.05">
        {ball('left_upper_leg')}
        <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" rgba="{BODY_RGBA}"/>
        <body name="left_lower_leg" pos="0 0 -0.42">
          {ball('left_lower_leg')}
          <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" rgba="{BODY_RGBA}"/>
          <body name="left_foot" pos="0 0 -0.40">
            {ball('left_foot')}
            <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" rgba="{ACCENT_RGBA}"/>
          </body>
        </body>
      </body>
      <body name="right_upper_leg" pos="-0.09 0 -0.05">
        {ball('right_upper_leg')}
        <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" rgba="{BODY_RGBA}"/>
        <body name="right_lower_leg" pos="0 0 -0.42">
          {ball('right_lower_leg')}
          <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" rgba="{BODY_RGBA}"/>
          <body name="right_foot" pos="0 0 -0.40">
            {ball('right_foot')}
            <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" rgba="{ACCENT_RGBA}"/>
          </body>
        </body>
      </body>
    </body>
    <body name="object_proxy" mocap="true" pos="0 -0.3 -5">
      {OBJECT_GEOM_XML}
    </body>
  </worldbody>
</mujoco>
"""

HUMANOID_XML_PATH = MUJOCO_DIR / "humanoid.xml"
HUMANOID_XML_PATH.write_text(HUMANOID_XML, encoding="utf-8")

# Validate the model. Loading alone needs no GL context, so this is safe.
mj_model = mujoco.MjModel.from_xml_string(HUMANOID_XML)
print(f"humanoid.xml OK  nq={mj_model.nq}  ball={len(MJ_BALL_ORDER)}  "
      f"hinge={len(MJ_HINGE_ORDER)}")

# --------------------- 7-B. Canonical rotations -> qpos ---------------------
# canonical (Y-up) -> MuJoCo (Z-up): conjugation by a +90 deg rotation about X
Q_C2MJ = np.array([math.cos(math.pi/4), math.sin(math.pi/4), 0.0, 0.0])
Q_C2MJ_INV = q_conj(Q_C2MJ)


def quat_c_to_mj(q):
    return q_normalize(q_mul(Q_C2MJ, q_mul(q, Q_C2MJ_INV)))


def vec_c_to_mj(v):
    return np.array([v[0], -v[2], v[1]])


# The qpos layout follows the XML tree order, so never assume it -- read the
# real joint addresses (jnt_qposadr) back from the compiled model.
NQ = int(mj_model.nq)
JOINT_LAYOUT = []
for joint_id in range(mj_model.njnt):
    JOINT_LAYOUT.append({
        "name": mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_JOINT, joint_id),
        "type": int(mj_model.jnt_type[joint_id]),
        "qpos_adr": int(mj_model.jnt_qposadr[joint_id]),
    })

QPOS = np.zeros((NUM_FRAMES, NQ), dtype=np.float32)
for fi in range(NUM_FRAMES):
    for joint in JOINT_LAYOUT:
        adr = joint["qpos_adr"]
        if joint["type"] == int(mujoco.mjtJoint.mjJNT_FREE):
            QPOS[fi, adr:adr + 3] = (vec_c_to_mj(HIPS_POS_C[fi])
                                     + np.array([0.0, 0.0, 0.93]))
            QPOS[fi, adr + 3:adr + 7] = quat_c_to_mj(BONE_ROT["hips"][fi])
        elif joint["type"] == int(mujoco.mjtJoint.mjJNT_BALL):
            QPOS[fi, adr:adr + 4] = quat_c_to_mj(BONE_ROT[joint["name"]][fi])
        else:  # hinge joints are the fingers
            side = "Left" if joint["name"].startswith("left") else "Right"
            finger = joint["name"].split("_")[-1]
            curl = float(CURLS[side][fi, FINGER_ORDER.index(finger)])
            QPOS[fi, adr] = float(np.clip(curl, 0.0, 2.2))

# Object proxy, converted into MuJoCo coordinates
OBJ_MJ = None
if OBJ_PROXY_C is not None:
    OBJ_MJ = np.stack([vec_c_to_mj(p) for p in OBJ_PROXY_C]).astype(np.float32)

# --------------------- 7-C. motion.npz + CSV + replay script ---------------------
MOTION_NPZ_PATH = MUJOCO_DIR / "motion.npz"
npz_payload = {
    "qpos": QPOS,
    "fps": np.array([EFFECTIVE_FPS], dtype=np.float32),
    "timestamp_sec": np.array([r["timestamp_sec"] for r in frames], dtype=np.float32),
    "joint_layout": np.array(json.dumps(JOINT_LAYOUT), dtype=object),
}
if OBJ_MJ is not None:
    npz_payload["object_pos"] = OBJ_MJ
np.savez_compressed(MOTION_NPZ_PATH, **npz_payload)

qpos_df = pd.DataFrame(QPOS, columns=[f"qpos_{i}" for i in range(NQ)])
qpos_df.insert(0, "timestamp_sec", [r["timestamp_sec"] for r in frames])
qpos_df.insert(0, "frame", range(NUM_FRAMES))
qpos_df.to_csv(DATASET_DIR / "adapters/mujoco_qpos.csv", index=False)

obj_rows = []
for fi in range(NUM_FRAMES):
    if OBJ_MJ is None:
        break
    obj_rows.append([fi, frames[fi]["timestamp_sec"], *np.round(OBJ_MJ[fi], 4),
                     OBJ_PROXY_SOURCE[fi]])
obj_traj_df = pd.DataFrame(obj_rows, columns=[
    "frame", "timestamp_sec", "x", "y", "z", "position_source"])
obj_traj_df.to_csv(MUJOCO_DIR / "object_trajectory.csv", index=False)
obj_traj_df.to_csv(DATASET_DIR / "adapters/mujoco_object_trajectory.csv", index=False)

REPLAY_SCRIPT = r"""'''Replay Common Behavior motion in the local MuJoCo viewer.

Usage:
    python replay_mujoco.py          # loop playback
    python replay_mujoco.py --once   # play once and hold the last frame

Keys (press inside the viewer window):
    SPACE : pause / resume
    R     : restart from frame 0
    , / . : step one frame backward / forward (while paused)
'''
import sys
import time
from pathlib import Path

import mujoco
import mujoco.viewer
import numpy as np

HERE = Path(__file__).parent
# Load the XML as a string. MjModel.from_xml_path opens the file down in C++
# and fails on non-ASCII paths, which is a common surprise on Windows.
model = mujoco.MjModel.from_xml_string(
    (HERE / "humanoid.xml").read_text(encoding="utf-8"))
data = mujoco.MjData(model)

archive = np.load(HERE / "motion.npz")
qpos = archive["qpos"]
fps = float(archive["fps"][0])
object_pos = archive["object_pos"] if "object_pos" in archive else None
total = len(qpos)
loop = "--once" not in sys.argv

print(f"frames={total}  fps={fps:.2f}  duration={total / fps:.2f}s  "
      f"loop={loop}")
print("keys: SPACE=pause/resume  R=restart  ,/.=step  (progress printed below)")

paused = False
frame = 0


def key_callback(keycode):
    '''Drive playback from the viewer keys (SPACE / R / , / .).'''
    global paused, frame
    key = chr(keycode) if 32 <= keycode < 127 else ""
    if keycode == 32:                     # SPACE
        paused = not paused
    elif key in ("r", "R"):
        frame = 0
    elif key == "." and paused:
        frame = min(frame + 1, total - 1)
    elif key == "," and paused:
        frame = max(frame - 1, 0)


with mujoco.viewer.launch_passive(model, data,
                                  key_callback=key_callback) as viewer:
    while viewer.is_running():
        step_start = time.time()
        index = min(frame, total - 1)
        data.qpos[:] = qpos[index]
        if object_pos is not None and model.nmocap > 0:
            data.mocap_pos[0] = object_pos[index]
        mujoco.mj_forward(model, data)
        viewer.sync()

        # Overwrite one progress line, so first/last frames are easy to spot
        marker = " <<< FIRST" if index == 0 else (
            " >>> LAST" if index == total - 1 else "")
        print(f"\rframe {index + 1:4d}/{total}  "
              f"t={index / fps:6.2f}s{'  [PAUSED]' if paused else ''}"
              f"{marker}          ", end="", flush=True)

        if not paused:
            if frame >= total - 1:
                if loop:
                    print("\n--- loop ---")
                    frame = 0
                else:
                    paused = True         # --once holds on the last frame
            else:
                frame += 1
        wait = 1.0 / fps - (time.time() - step_start)
        if wait > 0:
            time.sleep(wait)
print()
"""
(MUJOCO_DIR / "replay_mujoco.py").write_text(REPLAY_SCRIPT, encoding="utf-8")

(MUJOCO_DIR / "README_MUJOCO.md").write_text("""# MuJoCo Output

Model and motion are kept in separate files:

- `humanoid.xml` ... body / joint definition (MJCF)
- `motion.npz` .... qpos trajectory derived from Common Behavior Data
- `object_trajectory.csv` ... object proxy positions with `position_source`
- `replay_mujoco.py` ... viewer playback (kinematic replay, mj_forward only)
- `mujoco_simulation.mp4` ... rendered video (same trajectory as the viewer)

## Local playback (Windows)

```
pip install mujoco
py -m mujoco.viewer --mjcf="humanoid.xml"   # model only
py replay_mujoco.py                          # model + motion
```

Mode: Kinematic Replay (`data.qpos[:] = recorded_qpos[frame]` + `mj_forward`).
Physically correct contact/forces are out of scope for Demo 2.0.
""", encoding="utf-8")

print("Wrote:", MOTION_NPZ_PATH, "/ replay_mujoco.py / README_MUJOCO.md")

# --------------------- 7-D. Offscreen render in a subprocess ---------------------
# If MuJoCo cannot create a GL context it calls mju_error and aborts the whole
# process -- which in Colab kills the kernel. Rendering in a subprocess lets us
# fall back from egl to osmesa without losing the notebook state.
RENDER_WORKER_PATH = WORK_DIR / "mujoco_render_worker.py"
RENDER_CONFIG_PATH = WORK_DIR / "mujoco_render_config.json"
RAW_MUJOCO_PATH = WORK_DIR / "_mujoco_raw.mp4"

caption_meta = [{"frame": fi,
                 "time": f"{frames[fi]['timestamp_sec']:05.2f}",
                 "action": FRAME_PHASE[fi]["action"],
                 "phase": FRAME_PHASE[fi]["phase"],
                 "hand": FRAME_PHASE[fi]["hand"]} for fi in range(NUM_FRAMES)]

RENDER_WORKER_SOURCE = r'''
import json, os, sys
config = json.loads(open(sys.argv[1], encoding="utf-8").read())
os.environ["MUJOCO_GL"] = config["mujoco_gl"]
if config["mujoco_gl"] == "osmesa":
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"

import cv2
import numpy as np
import mujoco

model = mujoco.MjModel.from_xml_path(config["xml_path"])
data = mujoco.MjData(model)
archive = np.load(config["npz_path"], allow_pickle=True)
qpos = archive["qpos"]
object_pos = archive["object_pos"] if "object_pos" in archive else None

camera = mujoco.MjvCamera()
mujoco.mjv_defaultCamera(camera)
camera.azimuth = 90.0
camera.elevation = -10.0
camera.distance = config["camera_distance"]
camera.lookat[:] = config["camera_lookat"]

width, height = config["width"], config["height"]
print("creating renderer...", flush=True)
renderer = mujoco.Renderer(model, height=height, width=width)
print("renderer ready", flush=True)

writer = cv2.VideoWriter(config["out_path"],
                         cv2.VideoWriter_fourcc(*"mp4v"),
                         config["fps"], (width, height))
captions = config["captions"]
obj_label = config["object_label"]

def put(img, text, x, y, scale=0.55, color=(235, 235, 235)):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (20, 20, 20), 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, 1, cv2.LINE_AA)

for fi in range(len(qpos)):
    data.qpos[:] = qpos[fi]
    if object_pos is not None and model.nmocap > 0:
        data.mocap_pos[0] = object_pos[fi]
    mujoco.mj_forward(model, data)
    # Let the camera drift along with the pelvis
    camera.lookat[0] = float(data.qpos[0])
    renderer.update_scene(data, camera)
    frame = cv2.cvtColor(renderer.render(), cv2.COLOR_RGB2BGR)
    meta = captions[fi]
    put(frame, "MuJoCo Physics Simulation", 12, 26, 0.62)
    put(frame, "Source: Common Behavior Data   Mode: Kinematic Replay", 12, 48, 0.45)
    line3 = "Action: " + meta["action"] + "   Phase: " + meta["phase"]
    if obj_label:
        line3 += "   Object: " + obj_label
    put(frame, line3, 12, 70, 0.45)
    put(frame, "Frame: " + str(meta["frame"]) + "   Time: " + meta["time"] + " s",
        12, height - 14, 0.45)
    writer.write(frame)
    if fi % 60 == 0:
        print("frame", fi, "/", len(qpos), flush=True)

writer.release()
renderer.close()
print("done", flush=True)
'''
RENDER_WORKER_PATH.write_text(RENDER_WORKER_SOURCE, encoding="utf-8")

render_config = {
    "xml_path": str(HUMANOID_XML_PATH),
    "npz_path": str(MOTION_NPZ_PATH),
    "out_path": str(RAW_MUJOCO_PATH),
    "width": CONFIG["mujoco"]["render_width"],
    "height": CONFIG["mujoco"]["render_height"],
    "fps": EFFECTIVE_FPS,
    "camera_distance": 3.4,
    "camera_lookat": [0.0, 0.0, 0.90],
    "captions": caption_meta,
    "object_label": PRIMARY_TRACK["label"] if PRIMARY_TRACK else None,
    "mujoco_gl": None,
}

MUJOCO_MP4 = MUJOCO_DIR / "mujoco_simulation.mp4"
render_success = False
for gl_backend in (["egl", "osmesa"] if HAS_GPU else ["osmesa"]):
    render_config["mujoco_gl"] = gl_backend
    RENDER_CONFIG_PATH.write_text(json.dumps(render_config), encoding="utf-8")
    print(f"--- Rendering with MuJoCo (backend={gl_backend}, subprocess) ---")
    result = subprocess.run(
        [sys.executable, str(RENDER_WORKER_PATH), str(RENDER_CONFIG_PATH)],
        capture_output=True, text=True)
    if result.returncode == 0 and RAW_MUJOCO_PATH.exists():
        render_success = True
        break
    print(f"backend={gl_backend} failed (returncode={result.returncode})")
    print("--- stderr (tail) ---")
    print("\n".join(result.stderr.splitlines()[-8:]))

if render_success:
    encode_h264(RAW_MUJOCO_PATH, MUJOCO_MP4, EFFECTIVE_FPS)
    print("Wrote:", MUJOCO_MP4)
    show_video(MUJOCO_MP4)
else:
    print("MuJoCo rendering failed.")
    print("Try: switch to a GPU runtime and restart, or lower "
          "CONFIG['mujoco']['render_width'] to 640 and re-run [7].")


In [ ]:
# =====================================================================
# [8] VRM Animation (motion.vrma) export
#     Adapter #3. The same canonical bone rotations, this time mapped onto
#     VRM humanoid bones and written as glTF (GLB) + the VRMC_vrm_animation
#     extension -- playable in Unity on any VRM 1.0 avatar.
# =====================================================================
import struct

# ---- Common bone -> VRM humanoid bone mapping ----
COMMON_TO_VRM = {
    "hips": "hips", "spine": "spine", "chest": "chest",
    "neck": "neck", "head": "head",
    "left_upper_arm": "leftUpperArm", "left_lower_arm": "leftLowerArm",
    "left_hand": "leftHand",
    "right_upper_arm": "rightUpperArm", "right_lower_arm": "rightLowerArm",
    "right_hand": "rightHand",
    "left_upper_leg": "leftUpperLeg", "left_lower_leg": "leftLowerLeg",
    "left_foot": "leftFoot",
    "right_upper_leg": "rightUpperLeg", "right_lower_leg": "rightLowerLeg",
    "right_foot": "rightFoot",
}

# ---- VRMA skeleton (T-pose, glTF Y-up, person facing +Z) ----
VRM_NODE_DEFS = [
    ("hips", None, (0.0, 0.95, 0.0)),
    ("spine", "hips", (0.0, 0.10, 0.0)),
    ("chest", "spine", (0.0, 0.12, 0.0)),
    ("neck", "chest", (0.0, 0.20, 0.0)),
    ("head", "neck", (0.0, 0.06, 0.0)),
    ("leftUpperArm", "chest", (0.17, 0.14, 0.0)),
    ("leftLowerArm", "leftUpperArm", (0.26, 0.0, 0.0)),
    ("leftHand", "leftLowerArm", (0.25, 0.0, 0.0)),
    ("rightUpperArm", "chest", (-0.17, 0.14, 0.0)),
    ("rightLowerArm", "rightUpperArm", (-0.26, 0.0, 0.0)),
    ("rightHand", "rightLowerArm", (-0.25, 0.0, 0.0)),
    ("leftUpperLeg", "hips", (0.09, -0.05, 0.0)),
    ("leftLowerLeg", "leftUpperLeg", (0.0, -0.42, 0.0)),
    ("leftFoot", "leftLowerLeg", (0.0, -0.40, 0.0)),
    ("rightUpperLeg", "hips", (-0.09, -0.05, 0.0)),
    ("rightLowerLeg", "rightUpperLeg", (0.0, -0.42, 0.0)),
    ("rightFoot", "rightLowerLeg", (0.0, -0.40, 0.0)),
]

# ---- Finger bones (only where the mapping is stable enough to trust) ----
FINGER_SEGMENTS = {
    "Thumb": ["Metacarpal", "Proximal", "Distal"],
    "Index": ["Proximal", "Intermediate", "Distal"],
    "Middle": ["Proximal", "Intermediate", "Distal"],
    "Ring": ["Proximal", "Intermediate", "Distal"],
    "Little": ["Proximal", "Intermediate", "Distal"],
}
FINGER_Z_OFFSET = {"Thumb": 0.030, "Index": 0.020, "Middle": 0.0,
                   "Ring": -0.018, "Little": -0.034}
EXPORT_FINGERS = (CONFIG["vrm"]["export_fingers"]
                  and (HAND_VALID["Left"].any() or HAND_VALID["Right"].any()))
if EXPORT_FINGERS:
    for side, sign in [("left", 1.0), ("right", -1.0)]:
        hand_node = f"{side}Hand"
        for finger, segments in FINGER_SEGMENTS.items():
            parent = hand_node
            for si, segment in enumerate(segments):
                node_name = f"{side}{finger}{segment}"
                if si == 0:
                    translation = (sign * 0.035, 0.0, FINGER_Z_OFFSET[finger])
                else:
                    translation = (sign * 0.030, 0.0, 0.0)
                VRM_NODE_DEFS.append((node_name, parent, translation))
                parent = node_name

NODE_INDEX = {name: i for i, (name, _, _) in enumerate(VRM_NODE_DEFS)}

# ---- Assemble the animation tracks ----
times = np.array([r["timestamp_sec"] for r in frames], dtype=np.float32)


def axis_angle_quat_xyzw(axis, angle):
    axis = np.asarray(axis, float)
    s = math.sin(angle / 2)
    return np.array([axis[0]*s, axis[1]*s, axis[2]*s, math.cos(angle / 2)],
                    dtype=np.float32)


rotation_tracks = {}   # VRM node name -> (F, 4) quaternions, xyzw
for common, vrm in COMMON_TO_VRM.items():
    q = BONE_ROT[common]  # (F, 4) wxyz, canonical local -- already glTF space
    rotation_tracks[vrm] = np.stack(
        [q[:, 1], q[:, 2], q[:, 3], q[:, 0]], axis=1).astype(np.float32)

if EXPORT_FINGERS:
    for side_key, side_name, axis in [("Left", "left", (0, 0, -1.0)),
                                      ("Right", "right", (0, 0, 1.0))]:
        for finger_i, finger in enumerate(["Thumb", "Index", "Middle",
                                           "Ring", "Little"]):
            curls = CURLS[side_key][:, finger_i]  # rad, whole finger
            segments = FINGER_SEGMENTS[finger]
            per_segment = np.clip(curls / len(segments), 0.0, 1.6)
            track = np.stack([axis_angle_quat_xyzw(axis, a) for a in per_segment])
            for segment in segments:
                rotation_tracks[f"{side_name}{finger}{segment}"] = track

hips_translation = np.stack([
    HIPS_POS_C[:, 0], 0.95 + HIPS_POS_C[:, 1], HIPS_POS_C[:, 2]],
    axis=1).astype(np.float32)

# ---- Write the GLB container ----
binary_blob = bytearray()
buffer_views = []
accessors = []


def add_accessor(array, accessor_type, with_minmax=False):
    global binary_blob
    data = array.astype(np.float32).tobytes()
    offset = len(binary_blob)
    binary_blob += data
    while len(binary_blob) % 4:
        binary_blob += b"\x00"
    buffer_views.append({"buffer": 0, "byteOffset": offset,
                         "byteLength": len(data)})
    accessor = {"bufferView": len(buffer_views) - 1, "componentType": 5126,
                "count": len(array), "type": accessor_type}
    if with_minmax:
        flat = array.reshape(len(array), -1)
        accessor["min"] = [float(v) for v in flat.min(axis=0)]
        accessor["max"] = [float(v) for v in flat.max(axis=0)]
    accessors.append(accessor)
    return len(accessors) - 1


time_accessor = add_accessor(times.reshape(-1, 1), "SCALAR", with_minmax=True)

samplers, channels = [], []
for node_name, track in rotation_tracks.items():
    output_accessor = add_accessor(track, "VEC4")
    samplers.append({"input": time_accessor, "output": output_accessor,
                     "interpolation": "LINEAR"})
    channels.append({"sampler": len(samplers) - 1,
                     "target": {"node": NODE_INDEX[node_name],
                                "path": "rotation"}})
translation_accessor = add_accessor(hips_translation, "VEC3")
samplers.append({"input": time_accessor, "output": translation_accessor,
                 "interpolation": "LINEAR"})
channels.append({"sampler": len(samplers) - 1,
                 "target": {"node": NODE_INDEX["hips"], "path": "translation"}})

nodes = []
for name, parent, translation in VRM_NODE_DEFS:
    nodes.append({"name": name, "translation": list(translation)})
for i, (name, parent, _) in enumerate(VRM_NODE_DEFS):
    if parent is not None:
        nodes[NODE_INDEX[parent]].setdefault("children", []).append(i)

# Register only names that are valid VRM humanoid bones
valid_vrm_bones = set(COMMON_TO_VRM.values())
if EXPORT_FINGERS:
    for side in ["left", "right"]:
        for finger, segments in FINGER_SEGMENTS.items():
            for segment in segments:
                valid_vrm_bones.add(f"{side}{finger}{segment}")
human_bones = {name: {"node": NODE_INDEX[name]}
               for name in NODE_INDEX if name in valid_vrm_bones}

gltf_json = {
    "asset": {"version": "2.0",
              "generator": "human_behavior_demo_2_0 vrm_adapter"},
    "extensionsUsed": ["VRMC_vrm_animation"],
    "extensions": {"VRMC_vrm_animation": {
        "specVersion": "1.0",
        "humanoid": {"humanBones": human_bones}}},
    "scene": 0,
    "scenes": [{"nodes": [NODE_INDEX["hips"]]}],
    "nodes": nodes,
    "animations": [{"name": "human_behavior_motion",
                    "samplers": samplers, "channels": channels}],
    "buffers": [{"byteLength": len(binary_blob)}],
    "bufferViews": buffer_views,
    "accessors": accessors,
}

json_bytes = json.dumps(gltf_json, separators=(",", ":")).encode("utf-8")
while len(json_bytes) % 4:
    json_bytes += b" "
bin_bytes = bytes(binary_blob)
while len(bin_bytes) % 4:
    bin_bytes += b"\x00"

VRMA_PATH = UNITY_DIR / "motion.vrma"
total_length = 12 + 8 + len(json_bytes) + 8 + len(bin_bytes)
with open(VRMA_PATH, "wb") as fp:
    fp.write(struct.pack("<III", 0x46546C67, 2, total_length))
    fp.write(struct.pack("<II", len(json_bytes), 0x4E4F534A))  # 'JSON'
    fp.write(json_bytes)
    fp.write(struct.pack("<II", len(bin_bytes), 0x004E4942))   # 'BIN'
    fp.write(bin_bytes)

# adapters/vrm_bone_rotations.csv
rows = []
for fi in range(NUM_FRAMES):
    for common, vrm in COMMON_TO_VRM.items():
        x, y, z, w = rotation_tracks[vrm][fi]
        rows.append([fi, frames[fi]["timestamp_ms"], vrm,
                     round(float(x), 5), round(float(y), 5),
                     round(float(z), 5), round(float(w), 5)])
pd.DataFrame(rows, columns=["frame", "timestamp_ms", "vrm_bone",
                            "rot_x", "rot_y", "rot_z", "rot_w"]).to_csv(
    DATASET_DIR / "adapters/vrm_bone_rotations.csv", index=False)

# ---------------- Object trajectory for Unity (kinematic, position only) ----------------
# The object trajectory is part of the Common Behavior Data too, so it
# replays the same way the human motion does. The VRMA format only covers
# humanoids, so the object travels via a small JSON + C# script instead.
if OBJ_PROXY_C is not None:
    CYLINDER_LABELS = {"cup", "bottle", "wine glass", "vase", "bowl"}
    SPHERE_LABELS = {"sports ball", "ball", "apple", "orange", "frisbee"}
    obj_label = PRIMARY_TRACK["label"]
    obj_shape = ("cylinder" if obj_label in CYLINDER_LABELS
                 else "sphere" if obj_label in SPHERE_LABELS else "cube")
    trajectory_unity = {
        "label": obj_label,
        "shape": obj_shape,
        "fps": EFFECTIVE_FPS,
        "size": 0.12,
        "coordinate_note": ("canonical Y-up right-handed; the C# player "
                            "negates X to match UniVRM's VRM 1.0 import"),
        "frames": [{"t": round(frames[fi]["timestamp_sec"], 4),
                    "x": round(float(OBJ_PROXY_C[fi][0]), 4),
                    "y": round(float(OBJ_PROXY_C[fi][1]), 4),
                    "z": round(float(OBJ_PROXY_C[fi][2]), 4),
                    "source": OBJ_PROXY_SOURCE[fi]}
                   for fi in range(NUM_FRAMES)],
    }
    (UNITY_DIR / "object_trajectory_unity.json").write_text(
        json.dumps(trajectory_unity, indent=1), encoding="utf-8")

    OBJECT_PLAYER_CS = r'''using System;
using UnityEngine;

// Kinematic replay of a Common Behavior Data object trajectory in Unity.
// Usage: drop object_trajectory_unity.json and this file into Assets/, add
// this component to an empty GameObject, and assign trajectoryJson.
// Press Space during playback to re-sync with the avatar's loop start.
public class ObjectTrajectoryPlayer : MonoBehaviour
{
    [Serializable] public class TrajectoryFrame
    { public float t; public float x; public float y; public float z; public string source; }
    [Serializable] public class ObjectTrajectory
    { public string label; public string shape; public float fps; public float size;
      public TrajectoryFrame[] frames; }

    [Tooltip("Assign object_trajectory_unity.json")]
    public TextAsset trajectoryJson;
    public bool loop = true;
    [Tooltip("Matches UniVRM (VRM 1.0) X mirroring. Turn off if left/right looks flipped")]
    public bool mirrorX = true;
    [Tooltip("Playback time offset in seconds; also adjustable with the arrow keys")]
    public float timeOffset = 0f;

    ObjectTrajectory data;
    Transform target;
    float startTime;

    void Start()
    {
        if (trajectoryJson == null)
        { Debug.LogWarning("trajectoryJson is not assigned"); enabled = false; return; }
        data = JsonUtility.FromJson<ObjectTrajectory>(trajectoryJson.text);
        if (data == null || data.frames == null || data.frames.Length == 0)
        { Debug.LogWarning("Trajectory data is empty"); enabled = false; return; }

        PrimitiveType prim = data.shape == "cylinder" ? PrimitiveType.Cylinder
                           : data.shape == "sphere" ? PrimitiveType.Sphere
                           : PrimitiveType.Cube;
        GameObject go = GameObject.CreatePrimitive(prim);
        go.name = "TrackedObject_" + data.label;
        Destroy(go.GetComponent<Collider>());   // no physics: this is kinematic replay
        go.transform.localScale = Vector3.one * data.size;
        var renderer = go.GetComponent<Renderer>();
        renderer.material.color = new Color(0.85f, 0.30f, 0.30f);
        target = go.transform;
        startTime = Time.time;
    }

    void Update()
    {
        if (data == null) return;
        if (Input.GetKeyDown(KeyCode.Space)) startTime = Time.time;
        // Nudge the sync with the arrow keys: Left = earlier, Right = later
        if (Input.GetKeyDown(KeyCode.LeftArrow)) timeOffset += 0.1f;
        if (Input.GetKeyDown(KeyCode.RightArrow)) timeOffset -= 0.1f;

        float duration = data.frames[data.frames.Length - 1].t;
        float t = Time.time - startTime + timeOffset;
        if (t < 0f) t = loop ? t + duration : 0f;
        if (t > duration)
        {
            if (loop) { startTime = Time.time; t = 0f; }
            else t = duration;
        }
        // Interpolate between samples at playback time; the data stays discrete
        int i = 0;
        while (i < data.frames.Length - 2 && data.frames[i + 1].t < t) i++;
        TrajectoryFrame a = data.frames[i];
        TrajectoryFrame b = data.frames[Math.Min(i + 1, data.frames.Length - 1)];
        float u = b.t > a.t ? Mathf.Clamp01((t - a.t) / (b.t - a.t)) : 0f;
        target.position = Vector3.Lerp(ToUnity(a), ToUnity(b), u);
    }

    Vector3 ToUnity(TrajectoryFrame f)
        => new Vector3(mirrorX ? -f.x : f.x, f.y, f.z);

    void OnGUI()
    {
        // Sync status overlay -- delete this method to keep it out of a recording
        GUI.Label(new Rect(10, Screen.height - 44, 500, 20),
            "Object sync: Space=restart  Arrow L/R=offset " +
            timeOffset.ToString("+0.0;-0.0") + "s");
    }
}
'''
    (UNITY_DIR / "ObjectTrajectoryPlayer.cs").write_text(
        OBJECT_PLAYER_CS, encoding="utf-8")
    print("Wrote:", UNITY_DIR / "object_trajectory_unity.json",
          f"({NUM_FRAMES} frames, shape={obj_shape})")
    print("Wrote:", UNITY_DIR / "ObjectTrajectoryPlayer.cs")
else:
    print("No object trajectory: skipping object_trajectory_unity.json")

(UNITY_DIR / "README_UNITY.md").write_text("""# Unity / VRM Output

Avatar model and motion are kept separate:

- `avatar.vrm` ..... place a VRM 1.0 avatar here
  (e.g. VRM Consortium sample `Seed-san.vrm`)
- `motion.vrma` .... VRM Animation exported from Common Behavior Data
- `unity_vrm_animation.mp4` ... capture this locally in Unity

## Recommended: UniVRM SimpleVrma sample (no coding required)

1. Create a Unity project (2022.3 LTS or Unity 6, 3D template)
2. From https://github.com/vrm-c/UniVRM/releases download BOTH
   `UniVRM-0.1xx.x_xxxx.unitypackage` and
   `VRM_Samples-0.1xx.x_xxxx.unitypackage`, then import both via
   `Assets > Import Package > Custom Package...`
3. Open the scene `Assets/VRM10_Samples/SimpleVrma/SimpleVrma` and press Play
4. Use the on-screen UI to open `avatar.vrm` and `motion.vrma`
5. Turn OFF the `BoxMan` checkbox to hide the white skeleton preview
   (if BoxMan moves but the avatar does not, the issue is on the avatar
   side; if neither moves, the issue is on the vrma side)
6. Record the Game view with Unity Recorder
   (`Window > Package Manager` -> install Recorder ->
   `Window > General > Recorder` -> Movie / H.264 MP4)

Recording fps does NOT need to match the analysis fps - the comparison
cell aligns videos by timestamp. Just cover the full `duration_sec`
listed in `04_behavior_dataset/manifest.json`.

Unity performs no MediaPipe inference, motion smoothing, joint
calculation, or dataset generation - Load VRM, Load VRMA, Play only.

## Optional: replay the object trajectory (Level 1, kinematic)

The object trajectory is part of the Common Behavior Data, so it can be
replayed in Unity just like the human motion:

1. Copy `object_trajectory_unity.json` and `ObjectTrajectoryPlayer.cs`
   into the Unity project's `Assets/` folder
2. In the SimpleVrma scene (NOT in play mode), create an empty GameObject
   and add the `ObjectTrajectoryPlayer` component
3. Assign `object_trajectory_unity.json` to the `trajectoryJson` field
4. Press Play; a primitive matching the object class appears and follows
   the recorded trajectory. Press **Space** to restart the object motion
   at the moment the avatar's motion loops back to the start
5. If left/right looks mirrored against the avatar, turn OFF `mirrorX`

Position depth is monocular-estimated (see `source` per frame); rotation
and physical contact are out of scope for this level.

After capturing, rename the recording to `unity_vrm_animation.mp4`,
copy it back into this folder, and re-run the comparison cell to build
the three-screen video.
""", encoding="utf-8")

print(f"Wrote: {VRMA_PATH}  ({VRMA_PATH.stat().st_size/1024:.0f} KB, "
      f"{len(rotation_tracks)} bone tracks, fingers={EXPORT_FINGERS})")
print("Wrote:", UNITY_DIR / "README_UNITY.md")


## [8.5] Play it on a VRM avatar in Unity (outside Colab, optional)

Unity is **playback only** here — no inference, no motion computation. With
UniVRM's **SimpleVrma** sample scene you do not have to write a line of code.

1. Create a 3D project in Unity 2022.3 LTS (or Unity 6)
2. From [UniVRM Releases](https://github.com/vrm-c/UniVRM/releases) download
   **both** `UniVRM-0.1xx.x_xxxx.unitypackage` and
   `VRM_Samples-0.1xx.x_xxxx.unitypackage`, then import them via
   `Assets > Import Package > Custom Package...`
3. Open the scene `Assets/VRM10_Samples/SimpleVrma/SimpleVrma` and press ▶ Play
4. Load your avatar (`avatar.vrm`, e.g. the VRM Consortium's Seed-san) and the
   `motion.vrma` this notebook produced
5. Turn **BoxMan** off to hide the white skeleton preview.
   (Diagnostic: if BoxMan moves but the avatar does not, the problem is on the
   avatar side; if neither moves, it is on the vrma side.)
6. Record the Game view with Unity Recorder — install it from Package Manager,
   then `Window > General > Recorder`. **Recording at 30/60 fps is fine**:
   cell `[9]` aligns panes by timestamp, not by frame index. Just cover the
   `duration_sec` listed in `manifest.json`.
7. Rename the recording to `unity_vrm_animation.mp4`, put it back into
   `output/03_unity_vrm/`, and **re-run `[9]`** to complete the three-screen
   comparison.

Full details are written to `output/03_unity_vrm/README_UNITY.md` when `[8]` runs.


In [ ]:
# =====================================================================
# [9] Three-screen comparison video
#     The payoff shot: overlay, MuJoCo and Unity side by side, driven by the
#     SAME behavior data and aligned on the SAME timeline. Three different
#     renderers, one source of truth.
#     The Unity pane shows a placeholder until you capture it locally -- just
#     re-run this cell afterwards.
# =====================================================================

# Drop your locally captured Unity recording here and re-run to fill pane 3
UNITY_VIDEO_PATH = UNITY_DIR / "unity_vrm_animation.mp4"

BUILD_SCRIPT_SOURCE = r'''"""Build the three-screen motion comparison video.

Usage:
    python build_comparison_video.py comparison_config.json

The config lists the source videos, the common timeline (fps / duration),
and per-frame behavior captions taken from the Common Behavior Dataset.
All panes are sampled by *timestamp*, so videos with different fps are
still aligned to the same common timeline.
"""
import json
import sys
from pathlib import Path

import cv2
import numpy as np

PANE_HEIGHT = 420
TITLE_BAND = 52
CAPTION_BAND = 78


def put(image, text, x, y, scale=0.55, color=(235, 235, 235), thick=1):
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (15, 15, 15), thick + 2, cv2.LINE_AA)
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, thick, cv2.LINE_AA)


def open_pane(entry):
    path = entry.get("path")
    if path and Path(path).exists():
        cap = cv2.VideoCapture(path)
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            pane_width = max(2, int(round(width * PANE_HEIGHT / height)))
            pane_width -= pane_width % 2
            return {"cap": cap, "fps": fps, "width": pane_width,
                    "label": entry["label"], "last": None, "last_index": -1}
    pane_width = int(PANE_HEIGHT * 9 / 16)
    return {"cap": None, "fps": 30.0, "width": pane_width - pane_width % 2,
            "label": entry["label"], "last": None, "last_index": -1}


def pane_frame(pane, timestamp):
    if pane["cap"] is None:
        image = np.full((PANE_HEIGHT, pane["width"], 3), 45, np.uint8)
        put(image, "Pending", 20, PANE_HEIGHT // 2 - 10, 0.7, (160, 160, 160))
        put(image, "(capture in local Unity)", 20, PANE_HEIGHT // 2 + 18,
            0.45, (160, 160, 160))
        return image
    target = int(round(timestamp * pane["fps"]))
    while pane["last_index"] < target:
        success, frame = pane["cap"].read()
        if not success:
            break
        pane["last"] = frame
        pane["last_index"] += 1
    frame = pane["last"]
    if frame is None:
        return np.zeros((PANE_HEIGHT, pane["width"], 3), np.uint8)
    return cv2.resize(frame, (pane["width"], PANE_HEIGHT))


def main():
    config = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
    panes = [open_pane(entry) for entry in config["videos"]]
    fps = config["fps"]
    frame_count = config["frame_count"]
    captions = config["captions"]

    total_width = sum(p["width"] for p in panes)
    total_width -= total_width % 2
    total_height = TITLE_BAND + PANE_HEIGHT + CAPTION_BAND

    writer = cv2.VideoWriter(config["out_path"],
                             cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (total_width, total_height))
    for fi in range(frame_count):
        timestamp = fi / fps
        canvas = np.full((total_height, total_width, 3), 22, np.uint8)
        put(canvas, "Motion Comparison", 16, 34, 0.85, (250, 250, 250), 2)
        x = 0
        for pane in panes:
            image = pane_frame(pane, timestamp)
            height, width = image.shape[:2]
            canvas[TITLE_BAND:TITLE_BAND + height, x:x + width] = \
                image[:, :total_width - x] if x + width > total_width else image
            put(canvas, pane["label"], x + 10, TITLE_BAND + 24, 0.55)
            x += width
        meta = captions[min(fi, len(captions) - 1)]
        y0 = TITLE_BAND + PANE_HEIGHT
        put(canvas, "Action: %s   Phase: %s   Hand: %s" %
            (meta["action"], meta["phase"], meta["hand"]), 16, y0 + 30, 0.6)
        put(canvas, "Time: %s s   Frame: %d" % (meta["time"], meta["frame"]),
            16, y0 + 58, 0.6)
        writer.write(canvas)
    writer.release()
    for pane in panes:
        if pane["cap"] is not None:
            pane["cap"].release()
    print("done:", config["out_path"])


if __name__ == "__main__":
    main()
'''

BUILD_SCRIPT_PATH = COMPARISON_DIR / "build_comparison_video.py"
BUILD_SCRIPT_PATH.write_text(BUILD_SCRIPT_SOURCE, encoding="utf-8")

comparison_config = {
    "videos": [
        {"label": "MediaPipe Tracking", "path": str(OVERLAY_DIR / "mediapipe_overlay.mp4")},
        {"label": "MuJoCo Physics Simulation", "path": str(MUJOCO_DIR / "mujoco_simulation.mp4")},
        {"label": "Unity VRM Animation",
         "path": str(UNITY_VIDEO_PATH) if Path(UNITY_VIDEO_PATH).exists() else None},
    ],
    "fps": EFFECTIVE_FPS,
    "frame_count": NUM_FRAMES,
    "captions": [{"frame": fi,
                  "time": f"{frames[fi]['timestamp_sec']:05.2f}",
                  "action": FRAME_PHASE[fi]["action"],
                  "phase": FRAME_PHASE[fi]["phase"],
                  "hand": FRAME_PHASE[fi]["hand"]} for fi in range(NUM_FRAMES)],
    "out_path": str(WORK_DIR / "_comparison_raw.mp4"),
}
comparison_config_path = WORK_DIR / "comparison_config.json"
comparison_config_path.write_text(json.dumps(comparison_config), encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(BUILD_SCRIPT_PATH), str(comparison_config_path)],
    capture_output=True, text=True)
print(result.stdout.strip())
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Failed to build the comparison video")

COMPARISON_MP4 = COMPARISON_DIR / "motion_comparison.mp4"
encode_h264(WORK_DIR / "_comparison_raw.mp4", COMPARISON_MP4, EFFECTIVE_FPS)
print("Wrote:", COMPARISON_MP4)
if not Path(UNITY_VIDEO_PATH).exists():
    print("Note: the Unity pane is a placeholder. Capture "
          "unity_vrm_animation.mp4 in local Unity, put it at "
          f"{UNITY_VIDEO_PATH}, and re-run this cell.")
show_video(COMPARISON_MP4, width=760)


In [ ]:
# =====================================================================
# [10] Acceptance check + package
#     Verify every artifact, then bundle demo2_output_bundle.zip.
#     That ZIP is exactly what Demo B (the language-to-motion notebook)
#     consumes as one training episode -- keep it.
# =====================================================================
CHECKS = [
    ("Input: video loaded, fps/frames/duration read", SOURCE_VIDEO.exists()),
    ("MediaPipe: pose captured", COVERAGE["pose"] > 0),
    ("MediaPipe: left/right hand captured",
     COVERAGE["left_hand"] + COVERAGE["right_hand"] > 0),
    ("MediaPipe: face captured", COVERAGE["face"] > 0),
    ("MediaPipe: gesture captured", COVERAGE["gesture"] > 0),
    ("MediaPipe: object detection", COVERAGE["objects"] > 0),
    ("Objects: track ids assigned + trajectories saved",
     (DATASET_DIR / "objects/object_tracks.csv").exists() and len(TRACKS) > 0),
    ("Behavior: bone rotations generated",
     (DATASET_DIR / "human/bone_rotations.csv").exists()),
    ("Behavior: interaction candidates generated", len(INTERACTION_SEGMENTS) > 0),
    ("Behavior: metrics + quality generated",
     (DATASET_DIR / "quality/quality.json").exists()),
    ("Overlay: mediapipe_overlay.mp4",
     (OVERLAY_DIR / "mediapipe_overlay.mp4").exists()),
    ("MuJoCo: humanoid.xml and motion.npz kept separate",
     (MUJOCO_DIR / "humanoid.xml").exists() and (MUJOCO_DIR / "motion.npz").exists()),
    ("MuJoCo: replay_mujoco.py", (MUJOCO_DIR / "replay_mujoco.py").exists()),
    ("MuJoCo: mujoco_simulation.mp4",
     (MUJOCO_DIR / "mujoco_simulation.mp4").exists()),
    ("Unity: motion.vrma exported", (UNITY_DIR / "motion.vrma").exists()),
    ("Unity: unity_vrm_animation.mp4 (captured locally in Unity)",
     (UNITY_DIR / "unity_vrm_animation.mp4").exists()),
    ("Comparison: motion_comparison.mp4 + script",
     (COMPARISON_DIR / "motion_comparison.mp4").exists()
     and (COMPARISON_DIR / "build_comparison_video.py").exists()),
    ("Dataset: temporal captions (optional, needs a Gemini API key)",
     (DATASET_DIR / "captions/temporal_captions.json").exists()),
    ("Dataset: manifest / summary / README",
     all((DATASET_DIR / f).exists()
         for f in ["manifest.json", "behavior_summary.json", "README.md"])),
]

print("=== Acceptance criteria ===")
for label, ok in CHECKS:
    print(("  [x] " if ok else "  [ ] ") + label)

# ------------------------- Build the bundle ZIP -------------------------
BUNDLE_ROOT = WORK_DIR / "demo2_output_bundle"
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
BUNDLE_ROOT.mkdir(parents=True)
shutil.copytree(SOURCE_DIR, BUNDLE_ROOT / "source")
for name in ["01_mediapipe_overlay", "02_mujoco", "03_unity_vrm",
             "04_behavior_dataset", "05_comparison"]:
    shutil.copytree(OUTPUT_DIR / name, BUNDLE_ROOT / name)
shutil.copy(PROJECT_DIR / "README.md", BUNDLE_ROOT / "README.md")
shutil.copy(PROJECT_DIR / "config.json", BUNDLE_ROOT / "config.json")

ZIP_PATH = Path(shutil.make_archive(
    str(PROJECT_DIR / "demo2_output_bundle"), "zip", BUNDLE_ROOT))
print(f"\nBundle: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

try:
    from google.colab import files
    if ZIP_PATH.stat().st_size < 200 * 1024 * 1024:
        files.download(str(ZIP_PATH))
    else:
        print("Over 200 MB, skipping the automatic download -- "
              "grab it from the file pane on the left.")
except Exception:
    print("(not running in Colab, skipping the download)")

print("\nOne Source Video. One Common Behavior Dataset. Multiple Representations.")


## Tuning guide

Something not quite right? Edit `CONFIG` in `[2]`, then re-run from the cell listed.

| Symptom | Fix | Re-run from |
|---|---|---|
| Left and right hands are swapped | `mediapipe.mirror_handedness = True` | `[3]` |
| Analysis is slow / out of memory | lower `video.analysis_fps`, shorten `max_analysis_seconds` | `[3]` |
| The object is never picked up (wrong label) | add it to `object.target_labels` — see the "Detected objects" line at the end of `[3]` | `[4]` |
| The object is never picked up (low score) | lower `object.min_confidence` | `[3]` |
| One object splits into several tracks | normalise the label via `object.label_aliases`, or use `[3.5]` | `[4]` |
| No contact / grasp candidates | raise `behavior.contact_distance` (e.g. `0.45`) | `[4]` |
| Too many candidates | lower `behavior.contact_distance`, raise `behavior.reach_window` | `[4]` |
| Motion looks jittery | lower `cleaning.smoothing_alpha` (e.g. `0.35`) | `[4]` |
| Motion looks laggy | raise `cleaning.smoothing_alpha` (e.g. `0.7`) | `[4]` |
| The figure walks out of the MuJoCo frame | lower `mujoco.frame_world_width`, or set `use_image_translation = False` | `[4]` |
| MuJoCo rendering fails | set `mujoco.render_width = 640`; if it still fails, restart the runtime | `[7]` |
| No captions appear | check the secret is named `GEMINI_API_KEY` and notebook access is on | `[5.5]` |

---

### Where to go next

The `demo2_output_bundle.zip` from `[10]` is a complete behavior episode:
vision, language, motion and phase on one timeline. Feed a few of them into
**Demo B — Language to Motion** and the same representation becomes training
data, with generated behavior coming back out through these same adapters.
